# Use Case — Portfolio Heat Evaluation for the Real-Estate Agent

**Who this is for**  
Real-estate agents preparing a portfolio review with a client; portfolio managers and client advisors; secondarily REIT asset managers, private real-estate investors, and property operations leads.

**The scenario**  
You manage (or advise on) a 10-asset San Jose portfolio. The client wants to know which assets are at heat risk *today*, which represent the biggest *opportunity* if you treat them, and what those answers mean in dollars. Walking a building per day is not an option — you need a desk-first screening that gives you defensible numbers and slide-ready visuals before the meeting.

This notebook combines **your portfolio data** with **FortyGuard layers** to answer five questions:

1. **Where in the portfolio is hot, when, and by how much?**  ← 24-hour heatmap × portfolio
2. **Why are the hottest properties hot?**  ← satellite segmentation on top exposures
3. **What does the curb actually look like?**  ← street-view ground-truth on the #1 exposure
4. **What does tenant comfort look like through the day?**  ← environmental parameters
5. **What's the risk and the opportunity per asset, in dollars and tiers?**  ← composite scoring + business translation

> **Runs live against the API.** Add your `FORTYGUARD_API_KEY` to `.env` and place your portfolio CSV at `data/real_estate_san_jose_portfolio_sample.csv`. The `data/` directory is git-ignored and **not** shipped with the repo — bring your own input (see the schema below).

> **Bring your own portfolio.** Put a portfolio CSV at `data/real_estate_san_jose_portfolio_sample.csv` (the `data/` directory is git-ignored — not shipped). As long as the columns match (`property_id`, `name`, `type`, `year_built`, `sqft`, `market_value_musd`, `latitude`, `longitude`), everything downstream works; swap the path in Step 1 to use your own.

> **U.S. coverage only.** All FortyGuard endpoints operate over locations inside the United States. Swap the AOI to any U.S. city — coordinates outside the U.S. will return errors or empty responses.

> **Dates: 2021 to today.** `STUDY_DATE` must be on or after `2021-01-01` (the catalog's start) and no later than today; earlier or future dates fail at the heatmap call with a "no data available" error.

---

## Setup

In [ ]:
import sys, pathlib, time as _time
ROOT = pathlib.Path.cwd().parents[1]
sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv(ROOT / '.env')

import json
import random
import pandas as pd
import folium
import matplotlib.pyplot as plt
from shapely.geometry import Point, shape, mapping
from IPython.display import HTML, display

from fortyguard import FortyGuardClient
from fortyguard.exceptions import FortyGuardError
from fortyguard.samples import SAN_JOSE_POLYGON

# ── configuration ─────────────────────────────────────
STUDY_DATE         = '2024-10-02'         # change to the day you want to evaluate
STUDY_HOUR         = '14:00'              # design-peak afternoon — required by heatmap & satellite calls
GRANULARITY_M      = 80                   # heatmap resolution
TOP_N_TO_ENRICH    = 3                    # API budget for satellite / street view / env-params
BASELINE_C         = 24.0                 # comfortable ambient baseline
COOLING_KWH_PER_SF = 0.18                 # extra kWh per sf per °C above baseline
KWH_PRICE_USD      = 0.24
SLA_HI_C           = 32.0                 # tenant-comfort heat-index threshold
TEMP_C_SANITY      = (15.0, 55.0)         # F→C conversion sanity range

ASSET_TYPE_PALETTE = {
    'Office':       '#1f77b4',
    'Residential':  '#2ca02c',
    'Retail':       '#ff7f0e',
    'Mixed-Use':    '#9467bd',
    'Industrial':   '#8c564b',
}

# ── data paths — outputs and caches are organized by endpoint family ───
DATA            = ROOT / 'data'
PORTFOLIO_CSV   = DATA / 'real_estate_san_jose_portfolio_sample.csv'

HEATMAP_DIR     = DATA / 'heatmaps'
SAT_SEG_DIR     = DATA / 'satellite'
STREET_SEG_DIR  = DATA / 'street_view'
ENV_DIR         = DATA / 'env_params'
for d in (HEATMAP_DIR, SAT_SEG_DIR, STREET_SEG_DIR, ENV_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Default cache files — change the filename to load any captured run.
HEATMAP_GEOJSON = HEATMAP_DIR    / 'real_estate_san_jose_heatmap_sample_day_2024-10-02.geojson'
SATELLITE_JSON  = SAT_SEG_DIR    / 'real_estate_san_jose_satellite_segmentation_sample_day_2024-10-02.json'
STREETVIEW_JSON = STREET_SEG_DIR / 'real_estate_san_jose_street_view_segmentation_sample_day_2024-10-02.json'
ENV_PARAMS_JSON = ENV_DIR        / 'real_estate_san_jose_env_paramaters_sample_day_2024-10-02.json'
HEAT_INTEL_PDF  = DATA / 'real_estate_san_jose_heat_intelligence_sample_day_2024-10-02.pdf'

# Live client — only needed for Step 2a / 6a / 7a / 8a (cached cells run offline).
try:
    client = FortyGuardClient()
    HAVE_API = True
except FortyGuardError as exc:
    client = None
    HAVE_API = False
    print(f'[no API key] live cells will not work: {exc}')


# --- Quiet polling helper ----------------------------------------------------
# Each live API call prints exactly two lines: "Submitted ..." and "✓ ... completed."
# The API occasionally returns 403 "Unauthorized access" on the first poll before the
# activity is registered — we sleep briefly and retry up to a few times before giving up.
def submit_and_wait_quiet(client_method, label, *, poll_interval=8.0,
                          initial_delay=3.0, transient_403_retries=4,
                          failure_retries=2, timeout=600.0,
                          skip_on_failure=False, **kwargs):
    # Some analysis tasks (esp. satellite / street-view) intermittently fail or
    # hang on the backend, and some points have no imagery at all. Retry the whole
    # submit up to failure_retries times on a task-level failure; with
    # skip_on_failure=True, return None once retries/timeout are exhausted so a
    # caller loop can skip that point instead of aborting the whole step.
    def _giveup(exc):
        if skip_on_failure:
            print(f"  ⤼ {label} unavailable — skipping ({type(exc).__name__}).")
            return None
        print(f"  ✗ {label} failed: {exc}")
        raise exc
    for submit_attempt in range(failure_retries + 1):
        activity_id = client_method(wait=False, **kwargs)
        print(f"Submitted {label} → {activity_id}")

        if initial_delay:
            _time.sleep(initial_delay)

        for attempt in range(transient_403_retries + 1):
            try:
                result = client.wait_for(activity_id, poll_interval=poll_interval, timeout=timeout)
                print(f"  ✓ {label} completed.")
                return {"activity_id": activity_id, "result": result}
            except FortyGuardError as exc:
                msg = str(exc)
                is_403 = "-> 403" in msg or "Unauthorized access" in msg
                is_task_failure = " failed:" in msg and "-> " not in msg
                if is_403 and attempt < transient_403_retries:
                    back_off = 5 * (attempt + 1)
                    print(f"  ⏳ status returned 403 (transient); retrying in {back_off}s…")
                    _time.sleep(back_off)
                    continue
                if is_task_failure and submit_attempt < failure_retries:
                    back_off = 5 * (submit_attempt + 1)
                    print(f"  ↻ task failed (transient backend error); re-submitting in {back_off}s…")
                    _time.sleep(back_off)
                    break  # re-submit via the outer loop
                return _giveup(exc)
            except Exception as exc:
                return _giveup(exc)
    return _giveup(RuntimeError(f"{label}: exhausted all submit attempts"))


# Output bundle root — Step 10 will populate this folder.
OUTPUTS_ROOT = ROOT / 'outputs' / f'real_estate_{STUDY_DATE}'
OUTPUTS_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_CSV   = OUTPUTS_ROOT / 'portfolio_evaluation.csv'

print(f'STUDY_DATE={STUDY_DATE}  STUDY_HOUR={STUDY_HOUR}  TOP_N_TO_ENRICH={TOP_N_TO_ENRICH}')
print(f'Output bundle root: {OUTPUTS_ROOT.relative_to(ROOT)}')


---
## Step 1 — Load your portfolio

### What you are doing
Reading the portfolio CSV. Any columns may ride along — the workflow only needs `latitude` and `longitude` to do the geospatial work; the rest pass through to the final output so finance and ops see the asset IDs and values they already recognize.

### Why this matters
Starting from the operations system of record means the output carries *your* property IDs, *your* asset types, *your* square-footages — ready to paste into the client report or CRM.

In [ ]:
portfolio = pd.read_csv(PORTFOLIO_CSV)
type_counts = portfolio['type'].value_counts().to_dict()
print(f"Loaded {len(portfolio)} properties, "
      f"total value ${portfolio['market_value_musd'].sum():.0f}M, "
      f"total {portfolio['sqft'].sum():,} sqft")
print(f"  asset-type mix: {type_counts}")
portfolio

---
## Step 2a — Heat layer (via live API)

### What you are doing
Calling `client.create_heatmap` with `filter_type=3` (single day — covers the full 24 h; `start_time` is ignored). The response carries per-tile daily aggregates (`min/max/average_temperature`) in **°C**. The Enterprise API does not emit hourly `'00'..'23'` fields, so we use `max_temperature` as each tile's peak; peak_hour / diurnal_swing are unavailable (they'd need hourly data this API doesn't return).

### Why this matters
A single-day heatmap gives you both the daily peak (for ranking) and the diurnal pattern (for peak-hour and swing analysis) in one call. Daily peak is what determines whether tenants are uncomfortable in the worst part of the afternoon — that's the temperature every downstream signal should reflect.

In [ ]:
import numpy as np
import textwrap
from matplotlib.colors import LinearSegmentedColormap

# 12-stop spectral ramp (cool blue → hot red) — used by the summary card,
# the histogram, the colorbar, and the M1/M2/M3 maps.
TCM_COLORS = [
    '#2983ba', '#5aa4b2', '#88c4aa', '#b3e0a6', '#d1ecb0', '#f0f9ba',
    '#fff0ae', '#fed38c', '#fdb56a', '#f3854e', '#e54f35', '#d7191c',
]
TCM_CMAP = LinearSegmentedColormap.from_list('tcm', TCM_COLORS, N=256)

def temp_color(t, lo, hi):
    if t is None or lo is None or hi is None or hi == lo:
        return TCM_COLORS[0]
    frac = max(0.0, min(1.0, (float(t) - lo) / (hi - lo)))
    idx  = min(int(frac * len(TCM_COLORS)), len(TCM_COLORS) - 1)
    return TCM_COLORS[idx]


def show_heatmap_summary(temps, source_label):
    """Stats card + colored histogram + vertical colorbar — one figure summarizing
    the AOI temperature distribution. `temps` is the per-tile peak (°C) list."""
    temps = [t for t in temps if t is not None]
    if not temps:
        print('No tile temperatures to summarize.')
        return
    lo, hi = float(min(temps)), float(max(temps))
    mean   = float(sum(temps) / len(temps))

    wrapped_label = textwrap.fill(source_label, width=22) if source_label else ''
    n_label_lines = wrapped_label.count('\n') + 1

    fig = plt.figure(figsize=(12, 3.4 + 0.30 * max(0, n_label_lines - 1)),
                     constrained_layout=True)
    gs = fig.add_gridspec(1, 3, width_ratios=[1.4, 2.6, 0.20])

    ax0 = fig.add_subplot(gs[0, 0]); ax0.axis('off')
    ax0.text(0.0, 0.97, wrapped_label, transform=ax0.transAxes,
             fontsize=10.5, fontweight='bold', color='#222', va='top')
    subtitle_y = 0.97 - 0.11 * n_label_lines - 0.05
    ax0.text(0.0, subtitle_y, f'{len(temps):,} tiles',
             transform=ax0.transAxes,
             fontsize=10, color='#666', va='top')

    rows = [('min',  lo,   temp_color(lo,   lo, hi)),
            ('mean', mean, temp_color(mean, lo, hi)),
            ('max',  hi,   temp_color(hi,   lo, hi))]
    band_top    = subtitle_y - 0.10
    band_bottom = 0.05
    step        = (band_top - band_bottom) / max(len(rows) - 1, 1)
    rect_h      = min(0.14, step * 0.6)
    y = band_top
    for label, val, color in rows:
        ax0.text(0.0, y, label, transform=ax0.transAxes,
                 fontsize=10, color='#666', va='center', family='monospace')
        ax0.add_patch(plt.Rectangle((0.22, y - rect_h / 2), 0.10, rect_h,
                                    transform=ax0.transAxes,
                                    facecolor=color, edgecolor='#333', linewidth=0.6))
        ax0.text(0.37, y, f'{val:.2f} °C', transform=ax0.transAxes,
                 fontsize=13, fontweight='bold', color='#222', va='center',
                 family='monospace')
        y -= step

    ax1 = fig.add_subplot(gs[0, 1])
    _, edges, patches = ax1.hist(temps, bins=32, edgecolor='white', linewidth=0.4)
    for patch, edge_lo, edge_hi in zip(patches, edges[:-1], edges[1:]):
        patch.set_facecolor(temp_color((edge_lo + edge_hi) / 2, lo, hi))
    ax1.axvline(mean, color='#222', linestyle='--', linewidth=1.1, alpha=0.7)
    ax1.text(mean, 0.96, f'  mean {mean:.1f} °C',
             transform=ax1.get_xaxis_transform(),
             color='#222', fontsize=9, va='top')
    ax1.set_xlabel('Tile temperature (°C)')
    ax1.set_ylabel('Tile count')
    ax1.set_title('Temperature distribution across AOI')
    ax1.grid(axis='y', alpha=0.3)
    for spine in ('top', 'right'):
        ax1.spines[spine].set_visible(False)

    ax2 = fig.add_subplot(gs[0, 2])
    grad = np.linspace(lo, hi, 256).reshape(-1, 1)
    ax2.imshow(grad, aspect='auto', cmap=TCM_CMAP,
               extent=[0, 1, lo, hi], origin='lower')
    ax2.set_xticks([])
    ax2.yaxis.tick_right()
    ax2.set_ylabel('°C', rotation=0, labelpad=12, fontsize=9)

    plt.show()


def _as_c(f):
    return None if f is None else float(f)


if not HAVE_API:
    raise RuntimeError('Live mode requires a FORTYGUARD_API_KEY in .env. Run Step 2b instead.')

heatmap = submit_and_wait_quiet(
    client.create_heatmap,
    f'heatmap {STUDY_DATE}',
    polygon_aoi=SAN_JOSE_POLYGON,
    start_date=STUDY_DATE,
    start_time=STUDY_HOUR,
    filter_type=3,                # single day — daily aggregate per tile
    granularity=GRANULARITY_M,
)

map_data = heatmap['result'].get('map_data') or {}
features = map_data.get('features', []) if isinstance(map_data, dict) else []

# --- Persist the raw response under data/heatmaps/. ---
LIVE_HEATMAP_PATH = HEATMAP_DIR / f'heatmap_san_jose_{STUDY_DATE}_live.geojson'
with open(LIVE_HEATMAP_PATH, 'w', encoding='utf-8') as f:
    json.dump(map_data, f)
print(f'Saved raw heatmap → {LIVE_HEATMAP_PATH.relative_to(ROOT)}')

# --- Normalize features to (poly, hourly_c, peak_c, peak_h, min_c) tuples. ---
# filter_type=3 returns per-tile daily aggregates (min/max/average_temperature) in °C — no hourly.
# when present so Step 3 can derive peak_hour and diurnal_swing the same way the
# cache path does. Fall back to max_temperature only if hourly is missing.
tiles, minx, miny, maxx, maxy = [], 1e9, 1e9, -1e9, -1e9
for ft in features:
    poly  = shape(ft['geometry'])
    props = ft.get('properties', {}) or {}
    if all(f'{h:02d}' in props for h in range(24)):
        hourly_c = [_as_c(props[f'{h:02d}']) for h in range(24)]
        peak_c   = max(v for v in hourly_c if v is not None)
        peak_h   = next((i for i, v in enumerate(hourly_c) if v == peak_c), None)
        min_c    = min(v for v in hourly_c if v is not None)
    elif 'max_temperature' in props:
        peak_c = _as_c(props.get('max_temperature'))
        if peak_c is None:
            continue
        min_c    = _as_c(props.get('min_temperature')) or peak_c
        hourly_c = [peak_c] * 24
        peak_h   = None
    else:
        continue
    tiles.append((poly, hourly_c, peak_c, peak_h, min_c))
    x0, y0, x1, y1 = poly.bounds
    minx, miny = min(minx, x0), min(miny, y0)
    maxx, maxy = max(maxx, x1), max(maxy, y1)

aoi_bounds = (minx, miny, maxx, maxy)
peaks = [t[2] for t in tiles]
has_diurnal = any(t[3] is not None for t in tiles)
print(f'[live] {len(tiles):,} tiles, peak range {min(peaks):.1f}..{max(peaks):.1f} °C')
print(f'[live] hourly tile data: {"present" if has_diurnal else "absent"} '
      f'(peak_hour and diurnal_swing will be {"populated" if has_diurnal else "None"})')
show_heatmap_summary(peaks, f'Live API · {STUDY_DATE} (daily peak)')


def tile_for(lat, lon):
    p = Point(lon, lat)
    for t in tiles:
        if t[0].contains(p):
            return t
    return min(tiles, key=lambda t: t[0].centroid.distance(p))


---
## Step 2b — Or: load a cached heatmap (for testing)

### What you are doing
Loading a captured heatmap GeoJSON from `data/heatmaps/`. Each tile carries daily aggregates (`min`/`max`/`average_temperature`) in **°C**. We use `max_temperature` as each tile's peak `temperature`. The Enterprise API returns aggregates only — no hourly `'00'..'23'` — so diurnal-derived signals degrade to the daily peak.

### Why this matters
Use this path when iterating on the analysis without burning API credits. Step 2a writes new live captures into the same directory — change the filename in this cell to replay any captured run.

In [ ]:
import numpy as np
import textwrap
from matplotlib.colors import LinearSegmentedColormap

# 12-stop spectral ramp (cool blue → hot red).
TCM_COLORS = [
    '#2983ba', '#5aa4b2', '#88c4aa', '#b3e0a6', '#d1ecb0', '#f0f9ba',
    '#fff0ae', '#fed38c', '#fdb56a', '#f3854e', '#e54f35', '#d7191c',
]
TCM_CMAP = LinearSegmentedColormap.from_list('tcm', TCM_COLORS, N=256)

def temp_color(t, lo, hi):
    if t is None or lo is None or hi is None or hi == lo:
        return TCM_COLORS[0]
    frac = max(0.0, min(1.0, (float(t) - lo) / (hi - lo)))
    idx  = min(int(frac * len(TCM_COLORS)), len(TCM_COLORS) - 1)
    return TCM_COLORS[idx]


def show_heatmap_summary(temps, source_label):
    """Stats card + colored histogram + vertical colorbar — same visual as 2a."""
    temps = [t for t in temps if t is not None]
    if not temps:
        print('No tile temperatures to summarize.')
        return
    lo, hi = float(min(temps)), float(max(temps))
    mean   = float(sum(temps) / len(temps))

    wrapped_label = textwrap.fill(source_label, width=22) if source_label else ''
    n_label_lines = wrapped_label.count('\n') + 1

    fig = plt.figure(figsize=(12, 3.4 + 0.30 * max(0, n_label_lines - 1)),
                     constrained_layout=True)
    gs = fig.add_gridspec(1, 3, width_ratios=[1.4, 2.6, 0.20])

    ax0 = fig.add_subplot(gs[0, 0]); ax0.axis('off')
    ax0.text(0.0, 0.97, wrapped_label, transform=ax0.transAxes,
             fontsize=10.5, fontweight='bold', color='#222', va='top')
    subtitle_y = 0.97 - 0.11 * n_label_lines - 0.05
    ax0.text(0.0, subtitle_y, f'{len(temps):,} tiles',
             transform=ax0.transAxes,
             fontsize=10, color='#666', va='top')

    rows = [('min',  lo,   temp_color(lo,   lo, hi)),
            ('mean', mean, temp_color(mean, lo, hi)),
            ('max',  hi,   temp_color(hi,   lo, hi))]
    band_top    = subtitle_y - 0.10
    band_bottom = 0.05
    step        = (band_top - band_bottom) / max(len(rows) - 1, 1)
    rect_h      = min(0.14, step * 0.6)
    y = band_top
    for label, val, color in rows:
        ax0.text(0.0, y, label, transform=ax0.transAxes,
                 fontsize=10, color='#666', va='center', family='monospace')
        ax0.add_patch(plt.Rectangle((0.22, y - rect_h / 2), 0.10, rect_h,
                                    transform=ax0.transAxes,
                                    facecolor=color, edgecolor='#333', linewidth=0.6))
        ax0.text(0.37, y, f'{val:.2f} °C', transform=ax0.transAxes,
                 fontsize=13, fontweight='bold', color='#222', va='center',
                 family='monospace')
        y -= step

    ax1 = fig.add_subplot(gs[0, 1])
    _, edges, patches = ax1.hist(temps, bins=32, edgecolor='white', linewidth=0.4)
    for patch, edge_lo, edge_hi in zip(patches, edges[:-1], edges[1:]):
        patch.set_facecolor(temp_color((edge_lo + edge_hi) / 2, lo, hi))
    ax1.axvline(mean, color='#222', linestyle='--', linewidth=1.1, alpha=0.7)
    ax1.text(mean, 0.96, f'  mean {mean:.1f} °C',
             transform=ax1.get_xaxis_transform(),
             color='#222', fontsize=9, va='top')
    ax1.set_xlabel('Tile temperature (°C)')
    ax1.set_ylabel('Tile count')
    ax1.set_title('Temperature distribution across AOI')
    ax1.grid(axis='y', alpha=0.3)
    for spine in ('top', 'right'):
        ax1.spines[spine].set_visible(False)

    ax2 = fig.add_subplot(gs[0, 2])
    grad = np.linspace(lo, hi, 256).reshape(-1, 1)
    ax2.imshow(grad, aspect='auto', cmap=TCM_CMAP,
               extent=[0, 1, lo, hi], origin='lower')
    ax2.set_xticks([])
    ax2.yaxis.tick_right()
    ax2.set_ylabel('°C', rotation=0, labelpad=12, fontsize=9)

    plt.show()


def _as_c(f):
    return None if f is None else float(f)


with open(HEATMAP_GEOJSON, 'r', encoding='utf-8') as f:
    map_data = json.load(f)

# Enterprise API returns per-tile daily aggregates (min/max/average_temperature) in °C — no hourly '00'..'23'. Build (poly, hourly_c, peak, peak_h, min) tuples.
tiles, minx, miny, maxx, maxy = [], 1e9, 1e9, -1e9, -1e9
for ft in map_data.get('features', []):
    poly  = shape(ft['geometry'])
    props = ft.get('properties', {}) or {}
    if all(f'{h:02d}' in props for h in range(24)):
        hourly_c = [_as_c(props[f'{h:02d}']) for h in range(24)]
    elif 'max_temperature' in props:
        peak_c   = _as_c(props['max_temperature'])
        hourly_c = [peak_c] * 24
    else:
        continue
    peak_c = max(v for v in hourly_c if v is not None)
    peak_h = next((i for i, v in enumerate(hourly_c) if v == peak_c), None)
    min_c  = min(v for v in hourly_c if v is not None)
    tiles.append((poly, hourly_c, peak_c, peak_h, min_c))
    x0, y0, x1, y1 = poly.bounds
    minx, miny = min(minx, x0), min(miny, y0)
    maxx, maxy = max(maxx, x1), max(maxy, y1)

aoi_bounds = (minx, miny, maxx, maxy)
peaks = [t[2] for t in tiles]
lo, hi = min(peaks), max(peaks)
assert TEMP_C_SANITY[0] <= lo and hi <= TEMP_C_SANITY[1], \
    f'F→C sanity check failed: peak range {lo:.1f}..{hi:.1f} outside {TEMP_C_SANITY}'
print(f'[cached] {len(tiles)} tiles, peak temp range {lo:.1f}..{hi:.1f} °C')
print(f'[cached] AOI bounds (lon,lat): ({aoi_bounds[0]:.4f}, {aoi_bounds[1]:.4f}) → '
      f'({aoi_bounds[2]:.4f}, {aoi_bounds[3]:.4f})')
show_heatmap_summary(peaks, f'Cached · {HEATMAP_GEOJSON.name}')


def tile_for(lat, lon):
    p = Point(lon, lat)
    for t in tiles:
        if t[0].contains(p):
            return t
    return min(tiles, key=lambda t: t[0].centroid.distance(p))


### Visualize the heatmap

A spatial preview of the 24-hour **peak temperature** across every tile in the AOI. Each cell is shaded on a cool-blue → hot-red ramp by its daily maximum — hover any tile for the exact value. This is the layer the next steps join your portfolio against, so it pays to eyeball where the urban heat islands sit *before* the spatial join.

In [ ]:
# Visualize the average-temperature heatmap across the AOI using an
# **equal-interval classification** — same algorithm as the TS reference:
#     interval = (highest - least) / numberOfClasses
#     for i in 0..N-1:  class i covers [least + i*interval, least + (i+1)*interval)
#                       with color = colorRamp[i % len(colorRamp)]
# Every tile is then drawn with the color of the class its daily-average
# temperature falls into.
TCM_COLORS = [
    '#2983ba', '#5aa4b2', '#88c4aa', '#b3e0a6', '#d1ecb0', '#f0f9ba',
    '#fff0ae', '#fed38c', '#fdb56a', '#f3854e', '#e54f35', '#d7191c',
]
N_BINS = len(TCM_COLORS)

# Per-tile daily average over the 24 hourly values (hourly_c is index 1 of the tile tuple).
tile_avgs = [sum(t[1]) / len(t[1]) for t in tiles]
least, highest = min(tile_avgs), max(tile_avgs)
mean_t = sum(tile_avgs) / len(tile_avgs)
interval = (highest - least) / N_BINS if highest > least else 0.0

class_entries = [
    {'min':   least + i * interval,
     'max':   least + (i + 1) * interval,
     'color': TCM_COLORS[i % len(TCM_COLORS)]}
    for i in range(N_BINS)
]

def _class_color(t):
    """Match the TS reference: pick the class whose [min, max) contains t."""
    if t is None or interval == 0:
        return class_entries[0]['color']
    idx = int((t - least) / interval)        # floor → class index
    idx = max(0, min(idx, N_BINS - 1))       # the top edge collapses into the top class
    return class_entries[idx]['color']

heatmap_features = [{
    'type': 'Feature',
    'geometry': mapping(poly),
    'properties': {
        'avg_str': f"{avg_c:.2f} °C",
        'fillColor': _class_color(avg_c),
    },
} for (poly, _, _, _, _), avg_c in zip(tiles, tile_avgs)]

aoi_center = [(aoi_bounds[1] + aoi_bounds[3]) / 2,
              (aoi_bounds[0] + aoi_bounds[2]) / 2]
m_heat = folium.Map(location=aoi_center, zoom_start=13, tiles='cartodbpositron')

# Style each tile vivid + opaque, with a hairline stroke matched to the fill
# so adjacent cells seam cleanly into a solid raster (no basemap bleed,
# no dark gridlines from the polygon stroke).
folium.GeoJson(
    {'type': 'FeatureCollection', 'features': heatmap_features},
    style_function=lambda f: {
        'fillColor':   f['properties']['fillColor'],
        'color':       f['properties']['fillColor'],
        'weight':      0.6,
        'fillOpacity': 0.9,
        'opacity':     1.0,
    },
    tooltip=folium.GeoJsonTooltip(fields=['avg_str'], aliases=['Avg temp'], sticky=True),
).add_to(m_heat)
m_heat.fit_bounds([[aoi_bounds[1], aoi_bounds[0]], [aoi_bounds[3], aoi_bounds[2]]])

# Discrete legend — one row per equal-interval class, hottest on top.
rows = ''.join(
    f'<tr><td style="background:{c["color"]};width:18px;height:14px;'
    f'border:1px solid #888;"></td>'
    f'<td style="padding-left:8px;font-family:monospace;">'
    f'{c["min"]:.2f} – {c["max"]:.2f} °C</td></tr>'
    for c in reversed(class_entries)
)
legend_html = (
    '<div style="position:fixed;bottom:30px;left:30px;z-index:9999;'
    'background:white;padding:8px 12px;border:1px solid #888;font:12px sans-serif;">'
    '<b>Avg temperature (24 h)</b><br/>'
    f'<span style="color:#666;">equal-interval · {N_BINS} classes · '
    f'{interval:.2f} °C wide</span>'
    f'<table style="margin-top:4px;border-collapse:collapse;">{rows}</table></div>'
)
m_heat.get_root().html.add_child(folium.Element(legend_html))

print(f"Equal-interval classification on daily average: {N_BINS} classes of "
      f"width {interval:.2f} °C over {least:.2f}..{highest:.2f} °C "
      f"(AOI mean {mean_t:.2f} °C, {len(tiles):,} tiles)")
m_heat

---
## Step 3 — Diurnal temperature attach

### What you are doing
For each property, find the tile that contains it, copy off the **peak temperature**, **peak hour**, **min temperature**, **diurnal swing**, and the **AOI percentile** (this property's peak relative to all 16,507 city tiles). Now every property has an analysis-ready row.

### Why this matters
This is the moment your portfolio table becomes a *risk* table. Every downstream question — ranking, scoring, business translation — is a `groupby` or sort on this DataFrame.

In [ ]:
def _percentile_rank(value, sorted_values):
    lo, hi = 0, len(sorted_values)
    while lo < hi:
        mid = (lo + hi) // 2
        if sorted_values[mid] <= value: lo = mid + 1
        else: hi = mid
    return round(100.0 * lo / max(1, len(sorted_values)), 1)

aoi_peak_sorted = sorted(t[2] for t in tiles)

records = []
for _, r in portfolio.iterrows():
    poly, hourly_c, peak_c, peak_h, min_c = tile_for(r.latitude, r.longitude)
    records.append({
        'peak_temp_c'   : round(peak_c, 1),
        'peak_hour'     : peak_h,
        'min_temp_c'    : round(min_c, 1),
        'diurnal_swing_c': round(peak_c - min_c, 1),
        'aoi_percentile': _percentile_rank(peak_c, aoi_peak_sorted),
    })
portfolio = pd.concat([portfolio.reset_index(drop=True), pd.DataFrame(records)], axis=1)
portfolio = portfolio.sort_values('peak_temp_c', ascending=False).reset_index(drop=True)
portfolio.insert(0, 'temp_rank', portfolio.index + 1)

cols_t1 = ['temp_rank', 'property_id', 'name', 'type', 'sqft',
           'peak_temp_c', 'peak_hour', 'min_temp_c', 'diurnal_swing_c', 'aoi_percentile']
portfolio[cols_t1]

---
## Step 4 — Portfolio overview map (M1)

### What you are doing
Two layered views.

- **M1a** drops the full AOI heatmap (the equal-interval classes from Step 2, on daily-average °C) under the portfolio. Marker size scales with peak temperature; color encodes asset type.
- **M1b** isolates the spatial join: only the heatmap tiles that strictly contain a portfolio asset are drawn, with the same markers on top. Properties whose coordinates do not fall inside any tile (i.e., outside the AOI) are flagged.

### Why this matters
M1a is the slide-1 visual for the client meeting — before any score is computed, the agent can already see which assets sit in hot zones. M1b makes the join itself legible: every property is bound to one specific tile temperature, and that single number is the foundation for everything downstream.

In [ ]:
def _legend_html(palette):
    rows = ''.join(
        f'<tr><td style="background:{c};width:18px;"></td>'
        f'<td style="padding-left:6px;">{name}</td></tr>'
        for name, c in palette.items()
    )
    return (
        '<div style="position:fixed;bottom:30px;left:30px;z-index:9999;'
        'background:white;padding:8px 12px;border:1px solid #888;'
        'font:12px sans-serif;">'
        '<b>Asset type</b>'
        f'<table>{rows}</table>'
        '</div>'
    )

center = [portfolio['latitude'].mean(), portfolio['longitude'].mean()]
min_peak = portfolio['peak_temp_c'].min()

def _add_property_marker(m, p):
    folium.CircleMarker(
        location=[p.latitude, p.longitude],
        radius=4 + (p.peak_temp_c - min_peak) * 1.2,
        color='#000', weight=1.2,
        fill=True, fill_color=ASSET_TYPE_PALETTE.get(p['type'], '#888'),
        fill_opacity=0.95,
        popup=(f"<b>{p['name']}</b><br/>"
               f"{p['type']}, {p['sqft']:,} sqft, ${p['market_value_musd']}M<br/>"
               f"peak: {p.peak_temp_c:.1f}°C @ {format(int(p.peak_hour), '02d') + ':00' if pd.notna(p.peak_hour) else 'daily peak'}<br/>"
               f"AOI percentile: {p.aoi_percentile}"),
    ).add_to(m)

# ── M1a — full AOI heatmap with every portfolio point on top ──────────────────
m1a = folium.Map(location=center, zoom_start=13, tiles='cartodbpositron')
folium.GeoJson(
    {'type': 'FeatureCollection', 'features': heatmap_features},
    style_function=lambda f: {
        'fillColor':   f['properties']['fillColor'],
        'color':       f['properties']['fillColor'],
        'weight':      0.6,
        'fillOpacity': 0.75,
        'opacity':     1.0,
    },
    tooltip=folium.GeoJsonTooltip(fields=['avg_str'], aliases=['Avg temp'], sticky=True),
).add_to(m1a)
for _, p in portfolio.iterrows():
    _add_property_marker(m1a, p)
m1a.get_root().html.add_child(folium.Element(_legend_html(ASSET_TYPE_PALETTE)))
display(m1a)

# ── M1b — only the tiles a portfolio asset actually sits on ───────────────────
# Spatial-join footprint: each property maps to exactly one tile (or none, if
# its coordinates fall outside the AOI). We render only those tiles, colored
# by the same equal-interval classes, with the markers stacked on top.
def _strict_tile_for(lat, lon):
    """Return the tile that strictly contains the point, or None."""
    pt = Point(lon, lat)
    for t in tiles:
        if t[0].contains(pt):
            return t
    return None

joined_tiles, seen = [], set()
matched_props, unmatched_props = [], []
for _, p in portfolio.iterrows():
    t = _strict_tile_for(p.latitude, p.longitude)
    if t is None:
        unmatched_props.append(p)
        continue
    matched_props.append(p)
    poly, hourly_c, _, _, _ = t
    key = (round(poly.centroid.x, 6), round(poly.centroid.y, 6))
    if key in seen:
        continue
    seen.add(key)
    avg_c = sum(hourly_c) / len(hourly_c)
    joined_tiles.append({
        'type': 'Feature',
        'geometry': mapping(poly),
        'properties': {
            'avg_str':   f"{avg_c:.2f} °C",
            'fillColor': _class_color(avg_c),
        },
    })

print(f"M1b spatial join — {len(matched_props)} / {len(portfolio)} properties "
      f"sit on a heatmap tile ({len(joined_tiles)} unique tiles).")
if unmatched_props:
    names = ', '.join(f"{p['property_id']} ({p['name']})" for p in unmatched_props)
    print(f"  ⚠ Outside AOI / no containing tile: {names}")

m1b = folium.Map(location=center, zoom_start=14, tiles='cartodbpositron')
folium.GeoJson(
    {'type': 'FeatureCollection', 'features': joined_tiles},
    style_function=lambda f: {
        'fillColor':   f['properties']['fillColor'],
        'color':       '#000',
        'weight':      2.0,
        'fillOpacity': 0.9,
        'opacity':     1.0,
    },
    tooltip=folium.GeoJsonTooltip(fields=['avg_str'], aliases=['Tile avg'], sticky=True),
).add_to(m1b)
for p in matched_props:
    _add_property_marker(m1b, p)
m1b.fit_bounds([[portfolio['latitude'].min() - 0.005,
                 portfolio['longitude'].min() - 0.005],
                [portfolio['latitude'].max() + 0.005,
                 portfolio['longitude'].max() + 0.005]])
m1b.get_root().html.add_child(folium.Element(_legend_html(ASSET_TYPE_PALETTE)))
m1b

---
## Step 5 — Above-median hot exposures (M2)

### What you are doing
Two views, in order:

- **M2a** — AOI-wide heatmap classified on each tile's **24-h peak** (not the daily average from Step 2), with the full portfolio dropped on top. Every property is placed in the city's peak-temperature context.
- **M2b** — Drill-down to just the properties whose peak temperature is at or above the portfolio median, rendering only the heatmap tiles those assets sit on (M1b-style spatial join).

### Why this matters
M2a frames the conversation against the city — "your portfolio sits in *this* slice of San Jose at peak hour." M2b cuts the noise: the agent doesn't need ten talking points, they need a few. Together they are the slide-2 visual that says: "these are the assets we have to discuss, and here's how they compare to the city peak."

In [ ]:
# M2a — AOI-wide PEAK-temperature heatmap with the full portfolio on top.
# Same equal-interval scheme as Step 2, but classes are computed on each
# tile's 24-h max (not the daily average), so the legend tells you which
# slice of the city hits which peak.
tile_peaks    = [t[2] for t in tiles]
peak_least    = min(tile_peaks)
peak_highest  = max(tile_peaks)
peak_interval = (peak_highest - peak_least) / N_BINS if peak_highest > peak_least else 0.0
peak_classes  = [
    {'min':   peak_least + i * peak_interval,
     'max':   peak_least + (i + 1) * peak_interval,
     'color': TCM_COLORS[i % len(TCM_COLORS)]}
    for i in range(N_BINS)
]

def _peak_class_color(t):
    if t is None or peak_interval == 0:
        return peak_classes[0]['color']
    idx = int((t - peak_least) / peak_interval)
    return peak_classes[max(0, min(idx, N_BINS - 1))]['color']

peak_features = [{
    'type': 'Feature',
    'geometry': mapping(poly),
    'properties': {
        'peak_str':  f"{peak_c:.2f} °C peak",
        'fillColor': _peak_class_color(peak_c),
    },
} for poly, _, peak_c, _, _ in tiles]

m2a = folium.Map(location=center, tiles='cartodbpositron')
folium.GeoJson(
    {'type': 'FeatureCollection', 'features': peak_features},
    style_function=lambda f: {
        'fillColor':   f['properties']['fillColor'],
        'color':       f['properties']['fillColor'],
        'weight':      0.6,
        'fillOpacity': 0.85,
        'opacity':     1.0,
    },
    tooltip=folium.GeoJsonTooltip(fields=['peak_str'], aliases=['Tile peak'], sticky=True),
).add_to(m2a)
for _, p in portfolio.iterrows():
    folium.CircleMarker(
        location=[p.latitude, p.longitude],
        radius=4 + (p.peak_temp_c - min_peak) * 1.2,
        color='black', weight=1.2,
        fill=True, fill_color=ASSET_TYPE_PALETTE.get(p['type'], '#888'), fill_opacity=0.95,
        tooltip=f"#{int(p.temp_rank)} {p['property_id']} — {p.peak_temp_c:.1f}°C",
        popup=(f"<b>#{int(p.temp_rank)} {p['name']}</b><br/>"
               f"{p['type']}, {p['sqft']:,} sqft, ${p['market_value_musd']}M<br/>"
               f"peak: {p.peak_temp_c:.1f}°C @ {format(int(p.peak_hour), '02d') + ':00' if pd.notna(p.peak_hour) else 'daily peak'}<br/>"
               f"AOI percentile: {p.aoi_percentile}"),
    ).add_to(m2a)
m2a.fit_bounds([[aoi_bounds[1], aoi_bounds[0]], [aoi_bounds[3], aoi_bounds[2]]])

# Combined legend — peak-temp classes + asset-type swatches.
temp_rows_p = ''.join(
    f'<tr><td style="background:{c["color"]};width:18px;height:14px;'
    f'border:1px solid #888;"></td>'
    f'<td style="padding-left:8px;font-family:monospace;">'
    f'{c["min"]:.2f} – {c["max"]:.2f} °C</td></tr>'
    for c in reversed(peak_classes)
)
type_rows_p = ''.join(
    f'<tr><td style="background:{c};width:14px;height:14px;'
    f'border:1px solid #000;border-radius:50%;"></td>'
    f'<td style="padding-left:8px;">{name}</td></tr>'
    for name, c in ASSET_TYPE_PALETTE.items()
)
legend_html_p = (
    '<div style="position:fixed;bottom:30px;left:30px;z-index:9999;'
    'background:white;padding:8px 12px;border:1px solid #888;font:12px sans-serif;">'
    '<b>Tile peak temp (24 h)</b><br/>'
    f'<span style="color:#666;">equal-interval · {N_BINS} classes · '
    f'{peak_interval:.2f} °C wide · {peak_least:.2f}..{peak_highest:.2f} °C</span>'
    f'<table style="margin-top:4px;border-collapse:collapse;">{temp_rows_p}</table>'
    '<b style="display:block;margin-top:6px;">Asset type</b>'
    f'<table style="margin-top:4px;border-collapse:collapse;">{type_rows_p}</table>'
    '</div>'
)
m2a.get_root().html.add_child(folium.Element(legend_html_p))

print(f"M2a — AOI peak heatmap: {len(tiles):,} tiles, "
      f"peak range {peak_least:.2f}..{peak_highest:.2f} °C, "
      f"{N_BINS} classes of width {peak_interval:.2f} °C")
m2a

In [ ]:
median_peak = portfolio['peak_temp_c'].median()
hot = portfolio[portfolio['peak_temp_c'] >= median_peak].copy()

# Spatial join restricted to the hot subset: each above-median property
# pins to exactly one heatmap tile (the one its lat/lon lands inside).
# We render ONLY those tiles — same pattern as M1b — so the overlay
# answers "what tile does each hot asset sit on?" and nothing else.
def _strict_tile_for(lat, lon):
    pt = Point(lon, lat)
    for t in tiles:
        if t[0].contains(pt):
            return t
    return None

joined, seen = [], set()
matched_props, unmatched_props = [], []
for _, p in hot.iterrows():
    t = _strict_tile_for(p.latitude, p.longitude)
    if t is None:
        unmatched_props.append(p)
        continue
    matched_props.append(p)
    poly, _, peak_c, peak_h, _ = t
    key = (round(poly.centroid.x, 6), round(poly.centroid.y, 6))
    if key in seen:
        continue
    seen.add(key)
    joined.append({'poly': poly, 'peak_c': peak_c, 'peak_h': peak_h})

# Equal-interval classes recomputed on just the joined tiles so the
# legend describes the overlay (and not the full AOI).
joined_peaks = [j['peak_c'] for j in joined] or [median_peak]
least_h    = min(joined_peaks)
highest_h  = max(joined_peaks)
interval_h = (highest_h - least_h) / N_BINS if highest_h > least_h else 0.0
hot_classes = [
    {'min':   least_h + i * interval_h,
     'max':   least_h + (i + 1) * interval_h,
     'color': TCM_COLORS[i % len(TCM_COLORS)]}
    for i in range(N_BINS)
]

def _hot_class_color(t):
    if t is None or interval_h == 0:
        return hot_classes[-1]['color']
    idx = int((t - least_h) / interval_h)
    return hot_classes[max(0, min(idx, N_BINS - 1))]['color']

joined_features = [{
    'type': 'Feature',
    'geometry': mapping(j['poly']),
    'properties': {
        'peak_str':  f"{j['peak_c']:.2f} °C peak" + (f" @ {j['peak_h']:02d}:00" if j['peak_h'] is not None else " (daily peak)"),
        'fillColor': _hot_class_color(j['peak_c']),
    },
} for j in joined]

m2 = folium.Map(location=center, tiles='cartodbpositron')
folium.GeoJson(
    {'type': 'FeatureCollection', 'features': joined_features},
    style_function=lambda f: {
        'fillColor':   f['properties']['fillColor'],
        'color':       '#000',
        'weight':      1.5,
        'fillOpacity': 0.9,
        'opacity':     1.0,
    },
    tooltip=folium.GeoJsonTooltip(fields=['peak_str'], aliases=['Tile peak'], sticky=True),
).add_to(m2)
for p in matched_props:
    folium.CircleMarker(
        location=[p.latitude, p.longitude],
        radius=8, color='black', weight=1.2,
        fill=True, fill_color=ASSET_TYPE_PALETTE.get(p['type'], '#888'), fill_opacity=0.95,
        tooltip=f"#{int(p.temp_rank)} {p['property_id']} — {p.peak_temp_c:.1f}°C",
        popup=(f"<b>#{int(p.temp_rank)} {p['name']}</b><br/>"
               f"{p['type']}<br/>"
               f"peak: {p.peak_temp_c:.1f}°C @ {format(int(p.peak_hour), '02d') + ':00' if pd.notna(p.peak_hour) else 'daily peak'}<br/>"
               f"AOI percentile: {p.aoi_percentile}"),
    ).add_to(m2)

# Frame on the markers (with a small pad for context).
hot_lats = [p.latitude  for p in matched_props] or [hot['latitude'].mean()]
hot_lons = [p.longitude for p in matched_props] or [hot['longitude'].mean()]
lat_lo, lat_hi = min(hot_lats), max(hot_lats)
lon_lo, lon_hi = min(hot_lons), max(hot_lons)
pad_lat = max(0.004, (lat_hi - lat_lo) * 0.20)
pad_lon = max(0.004, (lon_hi - lon_lo) * 0.20)
m2.fit_bounds([[lat_lo - pad_lat, lon_lo - pad_lon],
               [lat_hi + pad_lat, lon_hi + pad_lon]])

# Combined legend — temperature classes (tile fill) + asset-type swatches (markers).
temp_rows = ''.join(
    f'<tr><td style="background:{c["color"]};width:18px;height:14px;'
    f'border:1px solid #888;"></td>'
    f'<td style="padding-left:8px;font-family:monospace;">'
    f'{c["min"]:.2f} – {c["max"]:.2f} °C</td></tr>'
    for c in reversed(hot_classes)
)
type_rows = ''.join(
    f'<tr><td style="background:{c};width:14px;height:14px;'
    f'border:1px solid #000;border-radius:50%;"></td>'
    f'<td style="padding-left:8px;">{name}</td></tr>'
    for name, c in ASSET_TYPE_PALETTE.items()
)
legend_html = (
    '<div style="position:fixed;bottom:30px;left:30px;z-index:9999;'
    'background:white;padding:8px 12px;border:1px solid #888;font:12px sans-serif;">'
    '<b>Tile peak temp (24 h)</b><br/>'
    f'<span style="color:#666;">portfolio-tile join · {N_BINS} equal-interval classes · '
    f'{interval_h:.2f} °C wide</span>'
    f'<table style="margin-top:4px;border-collapse:collapse;">{temp_rows}</table>'
    '<b style="display:block;margin-top:6px;">Asset type</b>'
    f'<table style="margin-top:4px;border-collapse:collapse;">{type_rows}</table>'
    '</div>'
)
m2.get_root().html.add_child(folium.Element(legend_html))

print(f"{len(hot)} above-median exposures (median peak {median_peak:.1f}°C); "
      f"{len(joined)} unique tiles after spatial join "
      f"(peak {least_h:.1f}–{highest_h:.1f}°C).")
if unmatched_props:
    names = ', '.join(f"{p['property_id']} ({p['name']})" for p in unmatched_props)
    print(f"  ⚠ Outside AOI / no containing tile: {names}")
display(m2)
hot[['temp_rank','property_id','name','type','latitude','longitude',
     'peak_temp_c','peak_hour','aoi_percentile']]

---
## Step 6a — Surface diagnosis (live, top-N)

### What you are doing
For each of the top-N hottest properties, calling `client.satellite_segmentation` with `filter_type=3` + `start_time=STUDY_HOUR`, persisting the raw response under `data/satellite/`, decoding the original + segmented imagery, and bucketing the class % into impervious / vegetation.

### Why this matters
Knowing a property is hot is not actionable on its own — *intervention selection depends on the cause*. Satellite segmentation tells you whether the heat driver is impervious surface (cool-pavement candidate), low vegetation (tree-planting candidate), or something else.

In [ ]:
import base64, io
from PIL import Image

IMPERV_KEYS = {'road', 'roads', 'pavement', 'building', 'buildings',
               'rooftop', 'rooftops', 'sidewalk', 'earth', 'bare', 'ground'}
VEGGIE_KEYS = {'vegetation', 'tree', 'trees', 'grass', 'greenery', 'park'}


def _bucket(segments, keys):
    total = 0.0
    for cls, pct in (segments or {}).items():
        if any(k in cls.lower() for k in keys):
            try: total += float(pct)
            except (TypeError, ValueError): pass
    return round(total, 1)


def _first_b64(value):
    if isinstance(value, list):
        return value[0] if value else None
    return value


def _decode_b64(b64_str):
    if not b64_str: return None
    if isinstance(b64_str, list): b64_str = b64_str[0]
    if b64_str.startswith('data:'): b64_str = b64_str.split(',', 1)[1]
    try:
        return Image.open(io.BytesIO(base64.b64decode(b64_str)))
    except Exception:
        return None


if not HAVE_API:
    raise RuntimeError('Live mode requires a FORTYGUARD_API_KEY. Run Step 6b instead.')

top_n = portfolio.head(TOP_N_TO_ENRICH).copy()
seg_data = {}
sat_imgs = {}

for _, r in top_n.iterrows():
    sat = submit_and_wait_quiet(
        client.satellite_segmentation,
        f"satellite #{int(r.temp_rank)} {r.property_id}",
        latitude=float(r.latitude), longitude=float(r.longitude),
        start_date=STUDY_DATE, start_time=STUDY_HOUR,
        filter_type=3, granularity=GRANULARITY_M,
    )
    res = sat['result']

    # Persist the raw response under data/satellite/.
    out_path = SAT_SEG_DIR / f'satellite_{r.property_id}_{STUDY_DATE}_live.json'
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(res, f)
    print(f'  saved → {out_path.relative_to(ROOT)}')

    seg_block = res.get('segmentation', {}) or {}
    seg_data[r.property_id] = seg_block.get('segments', {}) or {}
    sat_imgs[r.property_id] = {
        'orig': _first_b64(res.get('orignal_image') or res.get('original_image')),
        'seg':  seg_block.get('image_content'),
    }

top_n['impervious_pct'] = top_n['property_id'].map(
    lambda pid: _bucket(seg_data.get(pid), IMPERV_KEYS) if pid in seg_data else None)
top_n['vegetation_pct'] = top_n['property_id'].map(
    lambda pid: _bucket(seg_data.get(pid), VEGGIE_KEYS) if pid in seg_data else None)


# Per-property visualization — original + segmented imagery side by side.
for _, r in top_n.iterrows():
    pid = r.property_id
    if pid not in sat_imgs:
        continue
    imgs = sat_imgs[pid]
    orig_img = _decode_b64(imgs.get('orig'))
    seg_img  = _decode_b64(imgs.get('seg'))
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    if orig_img is not None: axes[0].imshow(orig_img)
    axes[0].set_title(f"#{int(r.temp_rank)} {pid} satellite ({r.latitude:.4f}, {r.longitude:.4f})")
    axes[0].axis('off')
    if seg_img is not None:  axes[1].imshow(seg_img)
    axes[1].set_title('Segmented image')
    axes[1].axis('off')
    plt.tight_layout(); plt.show()


# Stacked bar — top-N surface composition, parallel to the C2 chart.
enriched_rows = [(pid, segs) for pid, segs in seg_data.items() if segs]
if enriched_rows:
    classes = sorted({c for _, s in enriched_rows for c in s.keys()})
    fig, ax = plt.subplots(figsize=(8, max(2.5, 0.8 * len(enriched_rows))))
    bottoms = [0.0] * len(enriched_rows)
    pids    = [pid for pid, _ in enriched_rows]
    for c in classes:
        vals = [float(segs.get(c, 0.0)) for _, segs in enriched_rows]
        ax.barh(pids, vals, left=bottoms, label=c)
        bottoms = [b + v for b, v in zip(bottoms, vals)]
    ax.set_xlabel('% of surrounding scene')
    ax.set_title('C2 — Satellite surface composition (top exposures)')
    ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=8)
    plt.tight_layout(); plt.show()


# Map of all top-N properties.
fmap_sat = folium.Map(
    location=[top_n['latitude'].mean(), top_n['longitude'].mean()],
    zoom_start=14, tiles='cartodbpositron',
)
for _, r in top_n.iterrows():
    folium.Marker(
        location=[r.latitude, r.longitude],
        popup=(f"#{int(r.temp_rank)} {r.property_id} — {r['name']}<br/>"
               f"peak {r.peak_temp_c:.1f} °C<br/>"
               f"impervious {r.impervious_pct}% · vegetation {r.vegetation_pct}%"),
        icon=folium.Icon(color='red', icon='info-sign'),
    ).add_to(fmap_sat)
    folium.Circle(
        location=[r.latitude, r.longitude], radius=GRANULARITY_M / 2,
        color='red', fill=True, fill_opacity=0.15,
        popup=f'~{GRANULARITY_M}m tile',
    ).add_to(fmap_sat)
display(fmap_sat)

top_n[['temp_rank','property_id','name','peak_temp_c','impervious_pct','vegetation_pct']]


---
## Step 6b — Or: load cached satellite segmentation (for testing)

### What you are doing
For each top-N property, looking under `data/satellite/` for a per-property cache file (`<base>_<pid>.json`). Reads each file, decodes the imagery, and computes the same impervious / vegetation percentages.

### Why this matters
Use this path when iterating without burning API credits, or to replay specific captured runs by changing the filename pattern.

In [ ]:
import base64, io
from PIL import Image

IMPERV_KEYS = {'road', 'roads', 'pavement', 'building', 'buildings',
               'rooftop', 'rooftops', 'sidewalk', 'earth', 'bare', 'ground'}
VEGGIE_KEYS = {'vegetation', 'tree', 'trees', 'grass', 'greenery', 'park'}


def _bucket(segments, keys):
    total = 0.0
    for cls, pct in (segments or {}).items():
        if any(k in cls.lower() for k in keys):
            try: total += float(pct)
            except (TypeError, ValueError): pass
    return round(total, 1)


def _first_b64(value):
    if isinstance(value, list):
        return value[0] if value else None
    return value


def _decode_b64(b64_str):
    if not b64_str: return None
    if isinstance(b64_str, list): b64_str = b64_str[0]
    if b64_str.startswith('data:'): b64_str = b64_str.split(',', 1)[1]
    try:
        return Image.open(io.BytesIO(base64.b64decode(b64_str)))
    except Exception:
        return None


top_n = portfolio.head(TOP_N_TO_ENRICH).copy()
seg_data = {}
sat_imgs = {}
missing = []

for _, r in top_n.iterrows():
    pid = r.property_id
    cached_path = SATELLITE_JSON.with_name(
        SATELLITE_JSON.stem + f'_{pid.lower()}' + SATELLITE_JSON.suffix
    )
    if not cached_path.exists():
        missing.append(pid)
        continue
    with open(cached_path, 'r', encoding='utf-8') as f:
        sat_doc = json.load(f)
    seg_block = sat_doc.get('segmentation', {}) or {}
    seg_data[pid] = seg_block.get('segments', {}) or {}
    sat_imgs[pid] = {
        'orig': _first_b64(sat_doc.get('orignal_image') or sat_doc.get('original_image')),
        'seg':  seg_block.get('image_content'),
    }

if missing:
    print(f'⚠ Cached satellite missing for {missing}; run Step 6a to fetch live.')

top_n['impervious_pct'] = top_n['property_id'].map(
    lambda pid: _bucket(seg_data.get(pid), IMPERV_KEYS) if pid in seg_data else None)
top_n['vegetation_pct'] = top_n['property_id'].map(
    lambda pid: _bucket(seg_data.get(pid), VEGGIE_KEYS) if pid in seg_data else None)


# Per-property visualization — original + segmented imagery side by side.
for _, r in top_n.iterrows():
    pid = r.property_id
    if pid not in sat_imgs:
        continue
    imgs = sat_imgs[pid]
    orig_img = _decode_b64(imgs.get('orig'))
    seg_img  = _decode_b64(imgs.get('seg'))
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    if orig_img is not None: axes[0].imshow(orig_img)
    axes[0].set_title(f"#{int(r.temp_rank)} {pid} satellite (cached)")
    axes[0].axis('off')
    if seg_img is not None:  axes[1].imshow(seg_img)
    axes[1].set_title('Segmented image')
    axes[1].axis('off')
    plt.tight_layout(); plt.show()


# Stacked bar — top-N surface composition.
enriched_rows = [(pid, segs) for pid, segs in seg_data.items() if segs]
if enriched_rows:
    classes = sorted({c for _, s in enriched_rows for c in s.keys()})
    fig, ax = plt.subplots(figsize=(8, max(2.5, 0.8 * len(enriched_rows))))
    bottoms = [0.0] * len(enriched_rows)
    pids    = [pid for pid, _ in enriched_rows]
    for c in classes:
        vals = [float(segs.get(c, 0.0)) for _, segs in enriched_rows]
        ax.barh(pids, vals, left=bottoms, label=c)
        bottoms = [b + v for b, v in zip(bottoms, vals)]
    ax.set_xlabel('% of surrounding scene')
    ax.set_title('C2 — Satellite surface composition (top exposures)')
    ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=8)
    plt.tight_layout(); plt.show()

top_n[['temp_rank','property_id','name','peak_temp_c','impervious_pct','vegetation_pct']]


---
## Step 7a — Street-view ground truth (live, top-N)

### What you are doing
For each of the top-N hottest properties, calling `client.street_view_segmentation` for the front view and persisting the raw response under `data/street_view/`. We compute per-property scene composition.

### Why this matters
Satellite shows surroundings from above. Street view shows what someone walking up to the building actually sees — that's where you confirm shade-tree feasibility, building self-shading, sidewalk surface, and other ground-level details that drive intervention design.

In [ ]:
import base64, io
from PIL import Image


def _sv_payload(front):
    front = front or {}
    return {
        'orig'      : front.get('original_image'),
        'seg'       : front.get('segmented_image'),
        'segs'      : front.get('segments', {}) or {},
        'image_date': front.get('image_date', 'n/a'),
    }


def _decode(b64):
    if not b64: return None
    if isinstance(b64, list): b64 = b64[0]
    if b64.startswith('data:'): b64 = b64.split(',', 1)[1]
    try:
        return Image.open(io.BytesIO(base64.b64decode(b64)))
    except Exception:
        return None


if not HAVE_API:
    raise RuntimeError('Live mode requires a FORTYGUARD_API_KEY. Run Step 7b instead.')

sv_data = {}

for _, r in top_n.iterrows():
    sv_resp = submit_and_wait_quiet(
        client.street_view_segmentation,
        f"streetview #{int(r.temp_rank)} {r.property_id}",
        latitude=float(r.latitude), longitude=float(r.longitude),
        skip_on_failure=True, timeout=240,
    )
    if sv_resp is None:   # no street-view imagery / task failed — leave sv_* None
        continue
    res = sv_resp['result']

    # Persist the raw response.
    out_path = STREET_SEG_DIR / f'streetview_{r.property_id}_{STUDY_DATE}_live.json'
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(res, f)
    print(f'  saved → {out_path.relative_to(ROOT)}')

    sv_data[r.property_id] = _sv_payload(res.get('front'))


# "Property #1" = highest-rank property that actually has street-view imagery.
enriched_subset = top_n[top_n['property_id'].isin(sv_data.keys())] if sv_data else top_n
property_one = enriched_subset.iloc[0] if len(enriched_subset) else top_n.iloc[0]

if sv_data:
    print('Street-view loaded for: ' +
          ', '.join(f"{pid} ({d['image_date']})" for pid, d in sv_data.items()))


# Per-property visualization — original + segmented side by side.
for _, r in top_n.iterrows():
    pid = r.property_id
    d = sv_data.get(pid)
    if not d:
        continue
    orig_img = _decode(d.get('orig'))
    seg_img  = _decode(d.get('seg'))
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    if orig_img is not None: axes[0].imshow(orig_img)
    axes[0].set_title(f"#{int(r.temp_rank)} {pid} street view ({r.latitude:.4f}, {r.longitude:.4f})")
    axes[0].axis('off')
    if seg_img is not None:  axes[1].imshow(seg_img)
    axes[1].set_title('Segmentation')
    axes[1].axis('off')
    plt.tight_layout(); plt.show()


# C2b — stacked-bar street-view scene composition across top-N (parallel to C2 in Step 6).
sv_rows = [(r.property_id, sv_data[r.property_id]['segs'])
           for _, r in top_n.iterrows()
           if r.property_id in sv_data and sv_data[r.property_id].get('segs')]
if sv_rows:
    classes = sorted({c for _, s in sv_rows for c in s.keys()})
    fig, ax = plt.subplots(figsize=(8, max(2.5, 0.8 * len(sv_rows))))
    bottoms = [0.0] * len(sv_rows)
    pids    = [pid for pid, _ in sv_rows]
    for c in classes:
        vals = [float(segs.get(c, 0.0)) for _, segs in sv_rows]
        ax.barh(pids, vals, left=bottoms, label=c)
        bottoms = [b + v for b, v in zip(bottoms, vals)]
    ax.set_xlabel('% of street-view scene')
    ax.set_title('C2b — Street-view scene composition (front view)')
    ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=8)
    plt.tight_layout(); plt.show()


# Map of all top-N properties.
fmap_street = folium.Map(
    location=[top_n['latitude'].mean(), top_n['longitude'].mean()],
    zoom_start=15, tiles='cartodbpositron',
)
for _, r in top_n.iterrows():
    folium.Marker(
        location=[r.latitude, r.longitude],
        popup=(f"#{int(r.temp_rank)} {r.property_id} — {r['name']}<br/>"
               f"peak {r.peak_temp_c:.1f} °C"),
        icon=folium.Icon(color='red', icon='info-sign'),
    ).add_to(fmap_street)
display(fmap_street)


---
## Step 7b — Or: load cached street-view (for testing)

### What you are doing
Looks for a per-property cache file under `data/street_view/` (`<base>_<pid>.json`) and renders the same per-property imagery + scene composition.

### Why this matters
Use this path when iterating without burning API credits.

In [ ]:
import base64, io
from PIL import Image


def _sv_payload(front):
    front = front or {}
    return {
        'orig'      : front.get('original_image'),
        'seg'       : front.get('segmented_image'),
        'segs'      : front.get('segments', {}) or {},
        'image_date': front.get('image_date', 'n/a'),
    }


def _decode(b64):
    if not b64: return None
    if isinstance(b64, list): b64 = b64[0]
    if b64.startswith('data:'): b64 = b64.split(',', 1)[1]
    try:
        return Image.open(io.BytesIO(base64.b64decode(b64)))
    except Exception:
        return None


sv_data = {}
sv_missing = []

for _, r in top_n.iterrows():
    pid = r.property_id
    path = STREETVIEW_JSON.with_name(
        STREETVIEW_JSON.stem + f'_{pid.lower()}' + STREETVIEW_JSON.suffix
    )
    if not path.exists():
        sv_missing.append(pid)
        continue
    with open(path, 'r', encoding='utf-8') as f:
        sv_doc = json.load(f)
    sv_data[pid] = _sv_payload(sv_doc.get('front'))

if sv_missing:
    print(f'⚠ Cached street-view missing for {sv_missing}; run Step 7a to fetch live.')

# property_one — highest-rank property with street-view imagery.
enriched_subset = top_n[top_n['property_id'].isin(sv_data.keys())] if sv_data else top_n
property_one = enriched_subset.iloc[0] if len(enriched_subset) else top_n.iloc[0]


# Per-property visualization.
for _, r in top_n.iterrows():
    pid = r.property_id
    d = sv_data.get(pid)
    if not d:
        continue
    orig_img = _decode(d.get('orig'))
    seg_img  = _decode(d.get('seg'))
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    if orig_img is not None: axes[0].imshow(orig_img)
    axes[0].set_title(f"#{int(r.temp_rank)} {pid} street view (cached, imagery {d['image_date']})")
    axes[0].axis('off')
    if seg_img is not None:  axes[1].imshow(seg_img)
    axes[1].set_title('Segmentation')
    axes[1].axis('off')
    plt.tight_layout(); plt.show()


# C2b — stacked-bar street-view scene composition across top-N.
sv_rows = [(r.property_id, sv_data[r.property_id]['segs'])
           for _, r in top_n.iterrows()
           if r.property_id in sv_data and sv_data[r.property_id].get('segs')]
if sv_rows:
    classes = sorted({c for _, s in sv_rows for c in s.keys()})
    fig, ax = plt.subplots(figsize=(8, max(2.5, 0.8 * len(sv_rows))))
    bottoms = [0.0] * len(sv_rows)
    pids    = [pid for pid, _ in sv_rows]
    for c in classes:
        vals = [float(segs.get(c, 0.0)) for _, segs in sv_rows]
        ax.barh(pids, vals, left=bottoms, label=c)
        bottoms = [b + v for b, v in zip(bottoms, vals)]
    ax.set_xlabel('% of street-view scene')
    ax.set_title('C2b — Street-view scene composition (front view)')
    ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=8)
    plt.tight_layout(); plt.show()


---
## Step 8a — Diurnal driver profile (live, top-N)

### What you are doing
For each of the top-N hottest properties, calling `client.environmental_parameters` with `filter_type=3` (single day — covers the full 24 h; `start_time` is ignored) to get the full 24-hour diurnal series, persisting the raw response under `data/env_params/`, and computing peak heat-index / hours-above-SLA inside business hours (09:00–18:00).

### Why this matters
Heat index, apparent temperature, humidity, and solar irradiance through the day tell you whether the comfort problem is about absolute temperature, humidity stacking, or solar load — and which retrofit (HVAC capacity, ventilation, shading) addresses it. Tenant-comfort SLAs are written in terms of heat index, not dry-bulb temperature.

In [ ]:
def _slice_business_hours(values):
    return [values[h] for h in range(9, 19)]  # 09:00..18:00 inclusive


def _env_series(loc):
    p = (loc or {}).get('parameters', {}) or {}
    return {
        'heat_index_celsius'           : list(p.get('heat_index_celsius') or []),
        'apparent_temperature_celsius' : list(p.get('apparent_temperature_celsius') or []),
        'relative_humidity_percent'    : list(p.get('relative_humidity_percent') or []),
        # Solar may be a 24-hour series OR a daytime-aggregate dict (clear_sky.ghi/dni/dhi).
        'solar_irradiance'             : (loc or {}).get('solar_irradiance'),
    }


if not HAVE_API:
    raise RuntimeError('Live mode requires a FORTYGUARD_API_KEY. Run Step 8b instead.')

env_data = {}

for _, r in top_n.iterrows():
    env = submit_and_wait_quiet(
        client.environmental_parameters,
        f"env-params #{int(r.temp_rank)} {r.property_id}",
        latitude=float(r.latitude), longitude=float(r.longitude),
        temperature=float(r.peak_temp_c),
        start_date=STUDY_DATE, start_time=STUDY_HOUR,
        filter_type=3,                # single day — returns the full 24h diurnal series
    )
    res = env['result']

    # Persist the raw response.
    out_path = ENV_DIR / f'env_params_{r.property_id}_{STUDY_DATE}_live.json'
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(res, f)
    print(f'  saved → {out_path.relative_to(ROOT)}')

    locs = res.get('locations') or []
    if not locs:
        continue
    env_data[r.property_id] = _env_series(locs[0])


def _peak_solar(sol_raw):
    if isinstance(sol_raw, list) and sol_raw:
        return max(sol_raw)
    if isinstance(sol_raw, dict):
        return (sol_raw.get('clear_sky') or {}).get('ghi')
    return None


def _peak_metrics(s):
    hi   = s.get('heat_index_celsius') or []
    appt = s.get('apparent_temperature_celsius') or []
    business = _slice_business_hours(hi) if len(hi) >= 19 else hi
    return {
        'peak_heat_index_c'    : max(hi) if hi else None,
        'peak_apparent_temp_c' : max(appt) if appt else None,
        'hours_above_sla'      : sum(1 for v in business if v is not None and v > SLA_HI_C),
        'peak_solar_irradiance': _peak_solar(s.get('solar_irradiance')),
    }


metrics = {pid: _peak_metrics(s) for pid, s in env_data.items()}
for col in ('peak_heat_index_c', 'peak_apparent_temp_c', 'hours_above_sla', 'peak_solar_irradiance'):
    top_n[col] = top_n['property_id'].map(lambda p, _c=col: (metrics.get(p) or {}).get(_c))


# C3 — diurnal driver profile, one figure per property in temp_rank order.
def _draw_diurnal(ax_top, ax_bot, s, title):
    n = len(s.get('heat_index_celsius') or [])
    hours = list(range(n))
    if s.get('heat_index_celsius'):
        ax_top.plot(hours, s['heat_index_celsius'], label='Heat index (°C)', color='#d62728')
    if s.get('apparent_temperature_celsius'):
        ax_top.plot(hours, s['apparent_temperature_celsius'], label='Apparent temp (°C)', color='#ff7f0e')
    ax_top.axhline(SLA_HI_C, color='#888', linestyle='--', linewidth=1, label=f'SLA ({SLA_HI_C}°C)')
    ax_top.set_ylabel('°C')
    ax_top.set_xlabel('Hour of day')
    ax_top.legend(loc='upper left', fontsize=9)
    if s.get('relative_humidity_percent'):
        ax_rh = ax_top.twinx()
        ax_rh.plot(hours, s['relative_humidity_percent'], label='RH (%)', color='#1f77b4', alpha=0.6)
        ax_rh.set_ylabel('Relative humidity (%)', color='#1f77b4')
        ax_rh.legend(loc='upper right', fontsize=9)

    sol_raw = s.get('solar_irradiance')
    if isinstance(sol_raw, list) and sol_raw:
        ax_bot.plot(range(len(sol_raw)), sol_raw, color='#bcbd22')
        ax_bot.set_xlabel('Hour of day')
        ax_bot.set_ylabel('Solar irradiance (W/m²)')
    elif isinstance(sol_raw, dict) and sol_raw.get('clear_sky'):
        cs = sol_raw['clear_sky']
        ax_bot.bar(['GHI', 'DNI', 'DHI'],
                   [cs.get('ghi', 0), cs.get('dni', 0), cs.get('dhi', 0)],
                   color=['#bcbd22', '#e377c2', '#7f7f7f'])
        ax_bot.set_ylabel('W/m² (clear-sky daytime avg)')
        ax_bot.set_title('Clear-sky solar components')
    else:
        ax_bot.text(0.5, 0.5, 'No solar irradiance data', transform=ax_bot.transAxes,
                    ha='center', va='center', color='#888')
        ax_bot.set_xticks([]); ax_bot.set_yticks([])

    ax_top.set_title(title)


for _, r in top_n.iterrows():
    pid = r.property_id
    if pid not in env_data:
        continue
    fig, (ax_top, ax_bot) = plt.subplots(2, 1, figsize=(9, 6))
    _draw_diurnal(ax_top, ax_bot, env_data[pid],
                  f"C3 — Diurnal drivers — #{int(r.temp_rank)} {pid} ({r['name']})")
    plt.tight_layout(); plt.show()


# Map of top-N with peak driver readout.
fmap_env = folium.Map(
    location=[top_n['latitude'].mean(), top_n['longitude'].mean()],
    zoom_start=14, tiles='cartodbpositron',
)
for _, r in top_n.iterrows():
    folium.Marker(
        location=[r.latitude, r.longitude],
        popup=(f"#{int(r.temp_rank)} {r.property_id} — {r['name']}<br/>"
               f"peak HI {r.peak_heat_index_c} °C · hours > SLA: {r.hours_above_sla}"),
        icon=folium.Icon(color='red', icon='info-sign'),
    ).add_to(fmap_env)
display(fmap_env)

top_n[['temp_rank','property_id','name','peak_temp_c',
       'peak_heat_index_c','peak_apparent_temp_c','hours_above_sla']]


---
## Step 8b — Or: load cached env-parameters (for testing)

### What you are doing
For each top-N property, looks under `data/env_params/` for a per-property cache file (`<base>_<pid>.json`) and renders the same diurnal driver curves.

### Why this matters
Use this path when iterating without burning API credits.

In [ ]:
def _slice_business_hours(values):
    return [values[h] for h in range(9, 19)]  # 09:00..18:00 inclusive


def _env_series(loc):
    p = (loc or {}).get('parameters', {}) or {}
    return {
        'heat_index_celsius'           : list(p.get('heat_index_celsius') or []),
        'apparent_temperature_celsius' : list(p.get('apparent_temperature_celsius') or []),
        'relative_humidity_percent'    : list(p.get('relative_humidity_percent') or []),
        'solar_irradiance'             : (loc or {}).get('solar_irradiance'),
    }


env_data = {}
env_missing = []

for _, r in top_n.iterrows():
    pid = r.property_id
    path = ENV_PARAMS_JSON.with_name(
        ENV_PARAMS_JSON.stem + f'_{pid.lower()}' + ENV_PARAMS_JSON.suffix
    )
    if not path.exists():
        env_missing.append(pid)
        continue
    with open(path, 'r', encoding='utf-8') as f:
        env_doc = json.load(f)
    locs = env_doc.get('locations') or []
    if not locs:
        continue
    env_data[pid] = _env_series(locs[0])

if env_missing:
    print(f'⚠ Cached env-params missing for {env_missing}; run Step 8a to fetch live.')


def _peak_solar(sol_raw):
    if isinstance(sol_raw, list) and sol_raw:
        return max(sol_raw)
    if isinstance(sol_raw, dict):
        return (sol_raw.get('clear_sky') or {}).get('ghi')
    return None


def _peak_metrics(s):
    hi   = s.get('heat_index_celsius') or []
    appt = s.get('apparent_temperature_celsius') or []
    business = _slice_business_hours(hi) if len(hi) >= 19 else hi
    return {
        'peak_heat_index_c'    : max(hi) if hi else None,
        'peak_apparent_temp_c' : max(appt) if appt else None,
        'hours_above_sla'      : sum(1 for v in business if v is not None and v > SLA_HI_C),
        'peak_solar_irradiance': _peak_solar(s.get('solar_irradiance')),
    }


metrics = {pid: _peak_metrics(s) for pid, s in env_data.items()}
for col in ('peak_heat_index_c', 'peak_apparent_temp_c', 'hours_above_sla', 'peak_solar_irradiance'):
    top_n[col] = top_n['property_id'].map(lambda p, _c=col: (metrics.get(p) or {}).get(_c))


def _draw_diurnal(ax_top, ax_bot, s, title):
    n = len(s.get('heat_index_celsius') or [])
    hours = list(range(n))
    if s.get('heat_index_celsius'):
        ax_top.plot(hours, s['heat_index_celsius'], label='Heat index (°C)', color='#d62728')
    if s.get('apparent_temperature_celsius'):
        ax_top.plot(hours, s['apparent_temperature_celsius'], label='Apparent temp (°C)', color='#ff7f0e')
    ax_top.axhline(SLA_HI_C, color='#888', linestyle='--', linewidth=1, label=f'SLA ({SLA_HI_C}°C)')
    ax_top.set_ylabel('°C')
    ax_top.set_xlabel('Hour of day')
    ax_top.legend(loc='upper left', fontsize=9)
    if s.get('relative_humidity_percent'):
        ax_rh = ax_top.twinx()
        ax_rh.plot(hours, s['relative_humidity_percent'], label='RH (%)', color='#1f77b4', alpha=0.6)
        ax_rh.set_ylabel('Relative humidity (%)', color='#1f77b4')
        ax_rh.legend(loc='upper right', fontsize=9)

    sol_raw = s.get('solar_irradiance')
    if isinstance(sol_raw, list) and sol_raw:
        ax_bot.plot(range(len(sol_raw)), sol_raw, color='#bcbd22')
        ax_bot.set_xlabel('Hour of day')
        ax_bot.set_ylabel('Solar irradiance (W/m²)')
    elif isinstance(sol_raw, dict) and sol_raw.get('clear_sky'):
        cs = sol_raw['clear_sky']
        ax_bot.bar(['GHI', 'DNI', 'DHI'],
                   [cs.get('ghi', 0), cs.get('dni', 0), cs.get('dhi', 0)],
                   color=['#bcbd22', '#e377c2', '#7f7f7f'])
        ax_bot.set_ylabel('W/m² (clear-sky daytime avg)')
        ax_bot.set_title('Clear-sky solar components')
    else:
        ax_bot.text(0.5, 0.5, 'No solar irradiance data', transform=ax_bot.transAxes,
                    ha='center', va='center', color='#888')
        ax_bot.set_xticks([]); ax_bot.set_yticks([])

    ax_top.set_title(title)


for _, r in top_n.iterrows():
    pid = r.property_id
    if pid not in env_data:
        continue
    fig, (ax_top, ax_bot) = plt.subplots(2, 1, figsize=(9, 6))
    _draw_diurnal(ax_top, ax_bot, env_data[pid],
                  f"C3 — Diurnal drivers — #{int(r.temp_rank)} {pid} ({r['name']})")
    plt.tight_layout(); plt.show()

top_n[['temp_rank','property_id','name','peak_temp_c',
       'peak_heat_index_c','peak_apparent_temp_c','hours_above_sla']]


---
## Step 9 — Action brief

### What you are doing
For each of the top-N hottest properties, write a short plain-English brief that says **what's happening, why, and what to do next** — and stop there. Then one final map showing the same recommendation pinned to each property.

### Why this matters
Steps 1–8 produced everything an analyst needs: ranked exposures, surface composition, ground-truth imagery, diurnal driver curves. The real-estate operator on the receiving end does not need another score, table, or quadrant chart — they need to know **which building to fix first and what to do about it.** This step delivers that and nothing else.

Every recommendation cites a public intervention program (EPA Heat Island Reduction, USDA i-Tree, ASHRAE 90.1, ASHRAE 55, OSHA Heat Illness Prevention). The triggering condition is a measurement from Step 6 or Step 8 — never a constructed score.

In [ ]:
# Step 9 — Action brief: one plain-English paragraph per top-N property +
# one ranked map. Nothing else. Every recommendation cites a public program.

NOAA_EXTREME_CAUTION_C = 32.0   # NOAA NWS Heat Index 'Extreme Caution' threshold

def _pick_action(p):
    """Single most-relevant published intervention for this property's measurements."""
    imp = p.impervious_pct    if pd.notna(p.impervious_pct)    else None
    veg = p.vegetation_pct    if pd.notna(p.vegetation_pct)    else None
    hi  = p.peak_heat_index_c if pd.notna(p.peak_heat_index_c) else None

    if imp is not None and imp >= 70:
        return ("Cool-roof retrofit",
                "high-albedo roofing on a building surrounded by paved or built surface "
                "is the single highest-leverage move at this site",
                "EPA Heat Island Reduction · ASHRAE 90.1-2022 roof solar-reflectance")
    if veg is not None and veg < 20:
        return ("Shade-tree planting",
                "canopy expansion in a low-vegetation site lowers ambient temperature "
                "and shades the building envelope",
                "USDA Forest Service i-Tree · EPA Heat Island Reduction (Trees & Vegetation)")
    if hi is not None and hi >= NOAA_EXTREME_CAUTION_C:
        return ("HVAC capacity & indoor-comfort review",
                "peak heat index breaches NOAA's Extreme Caution threshold — verify "
                "the building still meets ASHRAE 55 indoor comfort during business hours",
                "ASHRAE 55-2020 · ASHRAE 90.1 energy compliance")
    return ("Annual monitoring only",
            "no published-threshold flags raised at this site this run",
            "—")

def _narrative(p, rank, total):
    parts = []
    parts.append("**The hottest property in your portfolio.**" if rank == 1
                 else f"**#{rank} of {total} hottest in your portfolio.**")
    pct_phrase = (f" (AOI {p.aoi_percentile:.0f}th percentile)"
                  if pd.notna(p.aoi_percentile) else "")
    if pd.notna(p.peak_hour):
        parts.append(f"Surface temperature peaks at **{p.peak_temp_c:.1f} °C** at "
                     f"{int(p.peak_hour):02d}:00{pct_phrase}.")
    else:
        parts.append(f"Surface temperature peaks at **{p.peak_temp_c:.1f} °C** "
                     f"(daily peak{pct_phrase}).")
    if pd.notna(p.impervious_pct) and pd.notna(p.vegetation_pct):
        parts.append(
            f"The surrounding scene is **{p.impervious_pct:.0f}% paved or built "
            f"and only {p.vegetation_pct:.0f}% vegetation** — that imbalance is "
            f"the primary heat driver."
        )
    if pd.notna(p.peak_heat_index_c) and p.peak_heat_index_c >= NOAA_EXTREME_CAUTION_C:
        hours = int(p.hours_above_sla) if pd.notna(p.hours_above_sla) else 0
        parts.append(
            f"Heat index reaches **{p.peak_heat_index_c:.1f} °C** — above NOAA's "
            f"Extreme Caution threshold — for {hours} business hours."
        )
    return " ".join(parts)

def _md_to_html(s):
    """Tiny markdown-bold → HTML conversion for the narrative."""
    out, bold = [], False
    i = 0
    while i < len(s):
        if s[i:i+2] == '**':
            out.append('</b>' if bold else '<b>')
            bold = not bold
            i += 2
        else:
            out.append(s[i])
            i += 1
    return ''.join(out)

top3 = top_n.sort_values('temp_rank').reset_index(drop=True)
total = len(top3)

print(f"Action brief — top {total} hottest properties.\n"
      "Every recommendation cites a public intervention program.\n")

for _, p in top3.iterrows():
    rank = int(p.temp_rank)
    action_label, action_why, action_cite = _pick_action(p)
    narrative_html = _md_to_html(_narrative(p, rank, total))

    # Explicit colors on every text node so the card stays readable in both
    # light and dark Jupyter / VS Code themes (themes can override inherited
    # text color but rarely override inline `color:` attributes).
    display(HTML(f'''
    <div style="border:1px solid #ccc;border-radius:6px;padding:18px 20px;margin:12px 0;
                font:13px/1.6 -apple-system,sans-serif;background:#ffffff;color:#1a1a1a">
      <div style="font:600 15px sans-serif;margin-bottom:10px;color:#0d0d0d">
        #{rank} · {p.property_id} — {p['name']}
        <span style="color:#555;font-weight:400">
          · {p['type']} · {int(p['sqft']):,} sqft · ${p['market_value_musd']}M
        </span>
      </div>
      <div style="margin:6px 0;color:#1a1a1a">{narrative_html}</div>
      <div style="margin-top:14px;padding:12px 16px;border-left:4px solid #d73027;
                  background:#fff5f5;border-radius:0 4px 4px 0;color:#1a1a1a">
        <div style="font:600 14px sans-serif;color:#0d0d0d">→ {action_label}</div>
        <div style="margin:4px 0;color:#333">{action_why.capitalize()}.</div>
        <div style="font-size:11px;color:#666;margin-top:6px">
          Reference: {action_cite}
        </div>
      </div>
    </div>
    '''))

# ── Final ranked map — top-N markers tooltip-labeled with the action ─────────
m3_lats = top3['latitude'].tolist()
m3_lons = top3['longitude'].tolist()
m3_center = ([sum(m3_lats) / len(m3_lats), sum(m3_lons) / len(m3_lons)]
             if m3_lats else center)
m3 = folium.Map(location=m3_center, zoom_start=14, tiles='cartodbpositron')

# Underlay — the heatmap tile each property sits on, faintly tinted.
seen_tiles = set()
for _, p in top3.iterrows():
    t = tile_for(p.latitude, p.longitude)
    if t is None:
        continue
    poly, _, peak_c, peak_h, _ = t
    key = (round(poly.centroid.x, 6), round(poly.centroid.y, 6))
    if key in seen_tiles:
        continue
    seen_tiles.add(key)
    folium.GeoJson(
        mapping(poly),
        style_function=lambda x: {
            'fillColor': '#d73027', 'color': '#d73027',
            'weight': 0.6, 'fillOpacity': 0.25,
        },
        tooltip=f"{peak_c:.1f}°C peak" + (f" @ {peak_h:02d}:00" if peak_h is not None else " (daily peak)"),
    ).add_to(m3)

# Markers — sized & colored by measured peak temperature; popup shows the action.
peak_lo = float(top3['peak_temp_c'].min())
peak_hi = float(top3['peak_temp_c'].max())
for _, p in top3.iterrows():
    action_label, _, action_cite = _pick_action(p)
    folium.CircleMarker(
        location=[p.latitude, p.longitude],
        radius=10 + (p.peak_temp_c - peak_lo) * 4,
        color='black', weight=1,
        fill=True,
        fill_color=temp_color(p.peak_temp_c, peak_lo, peak_hi),
        fill_opacity=0.92,
        tooltip=f"#{int(p.temp_rank)} {p['property_id']} — {action_label}",
        popup=(f"<b>#{int(p.temp_rank)} {p['name']}</b><br/>"
               f"peak: {p.peak_temp_c:.1f}°C @ {format(int(p.peak_hour), '02d') + ':00' if pd.notna(p.peak_hour) else 'daily peak'}<br/>"
               f"<b>→ {action_label}</b><br/>"
               f"<span style='color:#666;font-size:11px'>{action_cite}</span>"),
    ).add_to(m3)

if m3_lats:
    pad_lat = max(0.004, (max(m3_lats) - min(m3_lats)) * 0.30)
    pad_lon = max(0.004, (max(m3_lons) - min(m3_lons)) * 0.30)
    m3.fit_bounds([[min(m3_lats) - pad_lat, min(m3_lons) - pad_lon],
                   [max(m3_lats) + pad_lat, max(m3_lons) + pad_lon]])

m3.get_root().html.add_child(folium.Element(
    '<div style="position:fixed;bottom:30px;left:30px;z-index:9999;'
    'background:white;padding:8px 12px;border:1px solid #888;font:12px sans-serif;">'
    f'<b>Top-{total} hottest · recommended actions</b><br/>'
    '<span style="color:#666;">marker color &amp; size ∝ measured peak temperature</span><br/>'
    '<span style="color:#666;">click a marker for the recommendation</span>'
    '</div>'
))
display(m3)

---
## Step 10 — Package outputs (CSV + PDF + maps)

### What you are doing
Bundling everything the analysis produced into a single hand-off folder under `outputs/real_estate_<STUDY_DATE>/`:

- `portfolio_evaluation.csv` — the full per-property evaluation (already written by Step 9).
- `portfolio_report.pdf` — multi-page slide-deck-ready PDF with the heatmap summary, top-N satellite/street/env diagnoses, action briefs, and the priority table.
- `maps/*.html` — every interactive folium map (M1, M2, M3 + per-step diagnostic maps) saved as standalone HTML.

### Why this matters
Different stakeholders consume different formats. Asset managers want the CSV; the client deck wants a PDF; on-site teams want interactive maps. This step produces all three at once.

In [ ]:
import io, base64, json
from matplotlib.figure import Figure
from PIL import Image as PILImage

try:
    from reportlab.lib.pagesizes import letter
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.units import inch
    from reportlab.lib import colors
    from reportlab.lib.enums import TA_CENTER, TA_LEFT
    from reportlab.platypus import (
        SimpleDocTemplate, Paragraph, Spacer, Image as RLImage, Table, TableStyle,
        PageBreak, CondPageBreak, KeepTogether,
    )
except ImportError:
    raise RuntimeError(
        "reportlab is required for the PDF export. "
        "Install it with: pip install 'reportlab>=4.0.0' (or run pip install -r requirements.txt)."
    )

OUT_DIR = OUTPUTS_ROOT
(OUT_DIR / 'maps').mkdir(parents=True, exist_ok=True)


# --- 1. Save the portfolio evaluation CSV here. -----------------------------
csv_cols = [c for c in (
    'temp_rank','property_id','name','type','sqft','year_built','market_value_musd',
    'peak_temp_c','peak_hour','min_temp_c','diurnal_swing_c','aoi_percentile',
    'impervious_pct','vegetation_pct',
    'peak_heat_index_c','peak_apparent_temp_c','hours_above_sla','peak_solar_irradiance',
) if c in portfolio.columns]
csv_path = OUT_DIR / 'portfolio_evaluation.csv'
portfolio[csv_cols].to_csv(csv_path, index=False)
print(f'  ✓ {csv_path.relative_to(ROOT)}')


# --- 2. Save folium maps as standalone HTML. --------------------------------
maps = [
    ('m1_portfolio_overview.html', globals().get('m1')),
    ('m2_above_median.html',       globals().get('m2')),
    ('m3_priority_ranked.html',    globals().get('m3')),
    ('satellite_top_n.html',       globals().get('fmap_sat')),
    ('street_view_top_n.html',     globals().get('fmap_street')),
    ('env_params_top_n.html',      globals().get('fmap_env')),
]
for fname, m in maps:
    if m is None:
        continue
    m.save(str(OUT_DIR / 'maps' / fname))
    print(f'  ✓ maps/{fname}')


# --- 3. PDF report (ReportLab Platypus). ------------------------------------
PAGE_W, PAGE_H = letter
LEFT_MARGIN  = 0.6 * inch
RIGHT_MARGIN = 0.6 * inch
TOP_MARGIN   = 0.7 * inch
BOT_MARGIN   = 0.7 * inch
USABLE_W     = PAGE_W - LEFT_MARGIN - RIGHT_MARGIN

# Brand palette (FortyGuard).
BRAND_BLUE   = colors.HexColor('#0E4A8A')
BRAND_INK    = colors.HexColor('#1f2933')
BRAND_MUTED  = colors.HexColor('#5a6b7b')
BRAND_YELLOW = colors.HexColor('#FFD24D')

LOGO_PATH        = ROOT / 'assets' / 'fortyguard_logo.png'           # white wordmark — for blue backgrounds
LOGO_FOOTER_PATH = ROOT / 'assets' / 'fortyguard_logo_footer.png'    # two-tone blue — for white backgrounds
COVER_BG_PATH    = ROOT / 'assets' / 'cover_bg.png'                  # full-bleed cover background
LOGO_ASPECT        = 224 / 1208
LOGO_FOOTER_ASPECT = 68 / 364

REPORT_NAME = 'Real-Estate Portfolio Heat Evaluation'

_styles = getSampleStyleSheet()
S_H1    = ParagraphStyle('H1', parent=_styles['Heading1'],
                         fontName='Helvetica-Bold', fontSize=16, leading=20,
                         textColor=BRAND_BLUE, spaceBefore=4, spaceAfter=10)
S_H2    = ParagraphStyle('H2', parent=_styles['Heading2'],
                         fontName='Helvetica-Bold', fontSize=12, leading=15,
                         textColor=BRAND_INK, spaceBefore=4, spaceAfter=6)
S_BODY  = ParagraphStyle('Body', parent=_styles['BodyText'],
                         fontName='Helvetica', fontSize=10, leading=14,
                         textColor=BRAND_INK, spaceAfter=4)
S_BODY_W = ParagraphStyle('BodyW', parent=S_BODY, fontName='Helvetica', fontSize=8, leading=11)
S_CAP   = ParagraphStyle('Cap', parent=_styles['Italic'],
                         fontName='Helvetica-Oblique', fontSize=9, leading=12,
                         textColor=BRAND_MUTED, spaceAfter=8)
S_CONTACT = ParagraphStyle('Contact', parent=_styles['Normal'],
                           fontName='Helvetica', fontSize=9, leading=13,
                           textColor=BRAND_INK, spaceAfter=4)


def _h1(text):
    return Paragraph(str(text).upper().replace('&', '&amp;'), S_H1)


def _h2(text):
    return Paragraph(str(text).upper().replace('&', '&amp;'), S_H2)


def _fig_to_image(fig, max_width_inches=6.6, dpi=180):
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=dpi, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    buf.seek(0)
    img = PILImage.open(buf)
    iw, ih = img.size
    aspect = ih / iw
    width  = max_width_inches * inch
    return RLImage(buf, width=width, height=width * aspect)


def _pil_to_image(pil_img, max_width_inches=6.6):
    if pil_img is None:
        return None
    buf = io.BytesIO()
    pil_img.save(buf, format='PNG')
    buf.seek(0)
    iw, ih = pil_img.size
    aspect = ih / iw
    width  = max_width_inches * inch
    return RLImage(buf, width=width, height=width * aspect)


def _decode_b64(b64_str):
    if not b64_str:
        return None
    if isinstance(b64_str, list):
        b64_str = b64_str[0]
    if b64_str.startswith('data:'):
        b64_str = b64_str.split(',', 1)[1]
    try:
        return PILImage.open(io.BytesIO(base64.b64decode(b64_str)))
    except Exception:
        return None


def _wrap(s, style=S_BODY):
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return Paragraph('—', style)
    return Paragraph(str(s).replace('\n', '<br/>'), style)


TABLE_STYLE = TableStyle([
    ('BACKGROUND',    (0, 0),  (-1, 0),   BRAND_BLUE),
    ('TEXTCOLOR',     (0, 0),  (-1, 0),   colors.white),
    ('FONTNAME',      (0, 0),  (-1, 0),   'Helvetica-Bold'),
    ('FONTSIZE',      (0, 0),  (-1, 0),   9),
    ('ALIGN',         (0, 0),  (-1, 0),   'LEFT'),
    ('VALIGN',        (0, 0),  (-1, -1),  'MIDDLE'),
    ('LINEBELOW',     (0, 0),  (-1, 0),   0.6, BRAND_BLUE),
    ('LINEBELOW',     (0, -1), (-1, -1),  0.4, colors.HexColor('#cbd5e0')),
    ('FONTSIZE',      (0, 1),  (-1, -1),  8),
    ('LEFTPADDING',   (0, 0),  (-1, -1),  6),
    ('RIGHTPADDING',  (0, 0),  (-1, -1),  6),
    ('TOPPADDING',    (0, 0),  (-1, -1),  5),
    ('BOTTOMPADDING', (0, 0),  (-1, -1),  5),
    ('ROWBACKGROUNDS',(0, 1),  (-1, -1),  [colors.white, colors.HexColor('#f4f7fb')]),
])


def _draw_cover(canvas, doc):
    """Page 1: full-bleed bg.png, yellow accents, white wordmark at lower-left."""
    canvas.saveState()

    if COVER_BG_PATH.exists():
        canvas.drawImage(str(COVER_BG_PATH), 0, 0,
                         width=PAGE_W, height=PAGE_H,
                         preserveAspectRatio=False, mask='auto')
    else:
        canvas.setFillColor(BRAND_BLUE)
        canvas.rect(0, 0, PAGE_W, PAGE_H, stroke=0, fill=1)

    # Yellow pill containing the study date, top-left.
    pill_h = 0.34 * inch
    pill_w = 1.45 * inch
    pill_x = LEFT_MARGIN
    pill_y = PAGE_H - 1.05 * inch
    canvas.setFillColor(BRAND_YELLOW)
    canvas.roundRect(pill_x, pill_y, pill_w, pill_h, pill_h / 2,
                     stroke=0, fill=1)
    canvas.setFillColor(BRAND_INK)
    canvas.setFont('Helvetica-Bold', 11)
    canvas.drawCentredString(pill_x + pill_w / 2,
                             pill_y + pill_h / 2 - 0.04 * inch,
                             STUDY_DATE)

    # Title — large white uppercase, three short lines.
    canvas.setFillColor(colors.white)
    canvas.setFont('Helvetica-Bold', 38)
    title_top_y = PAGE_H - 1.8 * inch
    line_h      = 0.55 * inch
    canvas.drawString(LEFT_MARGIN, title_top_y - 0 * line_h, 'REAL-ESTATE')
    canvas.drawString(LEFT_MARGIN, title_top_y - 1 * line_h, 'PORTFOLIO HEAT')
    canvas.drawString(LEFT_MARGIN, title_top_y - 2 * line_h, 'EVALUATION')

    def _info_line(y, label, value):
        canvas.setFont('Helvetica-Bold', 11)
        canvas.setFillColor(BRAND_YELLOW)
        canvas.drawString(LEFT_MARGIN, y, label)
        label_w = canvas.stringWidth(label, 'Helvetica-Bold', 11)
        canvas.setFont('Helvetica', 11)
        canvas.setFillColor(colors.white)
        canvas.drawString(LEFT_MARGIN + label_w + 0.06 * inch, y, value)

    info_y = title_top_y - 2 * line_h - 0.55 * inch
    _info_line(info_y, 'AOI:',
               f' {len(tiles):,} tiles at {GRANULARITY_M} m  ·  '
               f'{len(portfolio)} properties  ·  top {TOP_N_TO_ENRICH} diagnosed in detail')

    aoi_peaks = [t[2] for t in tiles]
    info2_y = info_y - 0.28 * inch
    if aoi_peaks:
        aoi_min  = min(aoi_peaks)
        aoi_max  = max(aoi_peaks)
        aoi_mean = sum(aoi_peaks) / len(aoi_peaks)
        _info_line(info2_y, 'AOI peak temperature range:',
                   f' {aoi_min:.1f} – {aoi_max:.1f} °C  (mean {aoi_mean:.1f} °C)')

    upper_dots_y = (info2_y if aoi_peaks else info_y) - 0.40 * inch
    for i in range(3):
        canvas.setFillColor(BRAND_YELLOW)
        canvas.circle(LEFT_MARGIN + 0.10 * inch + i * 0.32 * inch, upper_dots_y,
                      0.10 * inch, stroke=0, fill=1)

    logo_w = 3.6 * inch
    logo_h = logo_w * LOGO_ASPECT
    logo_y = 1.2 * inch
    if LOGO_PATH.exists():
        canvas.drawImage(str(LOGO_PATH), LEFT_MARGIN, logo_y,
                         width=logo_w, height=logo_h, mask='auto')

    lower_dots_y = logo_y + logo_h + 0.28 * inch
    for i in range(3):
        canvas.setFillColor(BRAND_YELLOW)
        canvas.circle(LEFT_MARGIN + 0.10 * inch + i * 0.32 * inch, lower_dots_y,
                      0.10 * inch, stroke=0, fill=1)

    canvas.restoreState()


def _draw_body(canvas, doc):
    canvas.saveState()
    canvas.setStrokeColor(colors.HexColor('#cbd5e0'))
    canvas.setLineWidth(0.4)
    canvas.line(LEFT_MARGIN, 0.6 * inch, PAGE_W - RIGHT_MARGIN, 0.6 * inch)

    if LOGO_FOOTER_PATH.exists():
        foot_w = 0.85 * inch
        foot_h = foot_w * LOGO_FOOTER_ASPECT
        canvas.drawImage(str(LOGO_FOOTER_PATH),
                         LEFT_MARGIN, 0.32 * inch,
                         width=foot_w, height=foot_h, mask='auto')

    canvas.setFont('Helvetica', 8)
    canvas.setFillColor(BRAND_MUTED)
    canvas.drawString(LEFT_MARGIN + 1.0 * inch, 0.4 * inch,
                      f'{REPORT_NAME}  ·  {STUDY_DATE}')
    canvas.drawRightString(PAGE_W - RIGHT_MARGIN, 0.4 * inch, f'Page {doc.page}')
    canvas.restoreState()


# ── Matplotlib figure builders. ────────────────────────────────────────────
def _build_heatmap_distribution_fig():
    temps = [t[2] for t in tiles]
    if not temps:
        return None
    lo, hi, mean = float(min(temps)), float(max(temps)), float(sum(temps) / len(temps))

    fig = plt.figure(figsize=(11, 4.5), constrained_layout=True)
    gs = fig.add_gridspec(1, 3, width_ratios=[1.4, 2.6, 0.20])

    ax0 = fig.add_subplot(gs[0, 0]); ax0.axis('off')
    from matplotlib.patches import FancyBboxPatch
    ax0.add_patch(FancyBboxPatch(
        (-0.02, -0.02), 1.04, 1.04,
        transform=ax0.transAxes,
        boxstyle='round,pad=0.02,rounding_size=0.04',
        facecolor='white', edgecolor='#d8dee6', linewidth=0.8,
        clip_on=False, zorder=0,
    ))
    ax0.text(0.05, 0.92, f'Heatmap · {STUDY_DATE}', transform=ax0.transAxes,
             fontsize=12, fontweight='bold', va='top', zorder=2)
    ax0.text(0.05, 0.82, f'{len(temps):,} tiles', transform=ax0.transAxes,
             fontsize=10, color='#666', va='top', zorder=2)
    rows = [('min', lo), ('mean', mean), ('max', hi)]
    y = 0.62
    for label, val in rows:
        ax0.text(0.05, y, label, transform=ax0.transAxes,
                 fontsize=10, color='#666', va='center', family='monospace', zorder=2)
        ax0.add_patch(plt.Rectangle((0.27, y - 0.06), 0.10, 0.12,
                                    transform=ax0.transAxes,
                                    facecolor=temp_color(val, lo, hi),
                                    edgecolor='#333', linewidth=0.6, zorder=2))
        ax0.text(0.42, y, f'{val:.2f} °C', transform=ax0.transAxes,
                 fontsize=12, fontweight='bold', va='center', family='monospace', zorder=2)
        y -= 0.18

    ax1 = fig.add_subplot(gs[0, 1])
    _, edges, patches = ax1.hist(temps, bins=32, edgecolor='white', linewidth=0.4)
    for patch, e_lo, e_hi in zip(patches, edges[:-1], edges[1:]):
        patch.set_facecolor(temp_color((e_lo + e_hi) / 2, lo, hi))
    ax1.axvline(mean, color='#222', linestyle='--', linewidth=1.1, alpha=0.7)
    ax1.text(mean, 0.96, f'  mean {mean:.1f} °C',
             transform=ax1.get_xaxis_transform(),
             color='#222', fontsize=9, va='top')
    ax1.set_xlabel('Tile peak temperature (°C)')
    ax1.set_ylabel('Tile count')
    ax1.set_title('Daily peak temperature distribution across AOI')
    ax1.grid(axis='y', alpha=0.3)
    for sp in ('top', 'right'):
        ax1.spines[sp].set_visible(False)

    ax2 = fig.add_subplot(gs[0, 2])
    grad = np.linspace(lo, hi, 256).reshape(-1, 1)
    ax2.imshow(grad, aspect='auto', cmap=TCM_CMAP, extent=[0, 1, lo, hi], origin='lower')
    ax2.set_xticks([]); ax2.yaxis.tick_right()
    ax2.set_ylabel('°C', rotation=0, labelpad=12, fontsize=9)
    return fig


def _build_seg_breakdown_fig(segments, title, color='#3b8686'):
    if not segments:
        return None
    items   = sorted(segments.items(), key=lambda x: float(x[1]), reverse=True)
    classes = [k for k, _ in items]
    pcts    = [float(v) for _, v in items]
    fig, ax = plt.subplots(figsize=(8.5, max(2.5, 0.4 + 0.35 * len(classes))))
    ax.barh(classes, pcts, color=color, edgecolor='#333', linewidth=0.6)
    for i, p in enumerate(pcts):
        ax.text(p + max(pcts) * 0.015, i, f'{p:.1f}%',
                va='center', fontsize=9, fontweight='bold')
    ax.invert_yaxis()
    ax.set_xlabel('Coverage (%)')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.grid(axis='x', alpha=0.25, linestyle='--')
    for sp in ('top', 'right'):
        ax.spines[sp].set_visible(False)
    plt.tight_layout()
    return fig


def _build_top_n_surface_comp_fig(seg_dict, title):
    rows = [(pid, segs) for pid, segs in (seg_dict or {}).items() if segs]
    if not rows:
        return None
    classes = sorted({c for _, s in rows for c in s.keys()})
    fig, ax = plt.subplots(figsize=(9, max(2.5, 0.7 * len(rows))))
    bottoms = [0.0] * len(rows)
    pids = [pid for pid, _ in rows]
    for c in classes:
        vals = [float(segs.get(c, 0.0)) for _, segs in rows]
        ax.barh(pids, vals, left=bottoms, label=c)
        bottoms = [b + v for b, v in zip(bottoms, vals)]
    ax.set_xlabel('% of scene')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=8)
    plt.tight_layout()
    return fig


def _build_diurnal_fig(s, title):
    if not s or not s.get('heat_index_celsius'):
        return None
    fig, (ax_top, ax_bot) = plt.subplots(2, 1, figsize=(9, 6))
    _draw_diurnal(ax_top, ax_bot, s, title)
    plt.tight_layout()
    return fig


# ── Build flowables. ───────────────────────────────────────────────────────
flowables = []

# Cover drawn on canvas; PageBreak advances to body.
flowables.append(PageBreak())

# Heatmap distribution
flowables.append(_h1('AOI peak-temperature distribution'))
flowables.append(Paragraph(
    'Per-tile daily peak across the AOI. The dashed line marks the AOI mean; '
    'the histogram is colored on the same spectral ramp as every map and chart in this report.',
    S_BODY))
hm_fig = _build_heatmap_distribution_fig()
if hm_fig is not None:
    flowables.append(Spacer(1, 0.1 * inch))
    flowables.append(_fig_to_image(hm_fig, max_width_inches=USABLE_W / inch))
flowables.append(Spacer(1, 0.35 * inch))
flowables.append(CondPageBreak(3.5 * inch))

# Top-N at a glance
flowables.append(_h1(f'Top {TOP_N_TO_ENRICH} hottest properties — at a glance'))
flowables.append(Paragraph(
    'Daily peak temperature, surface diagnosis (% impervious vs. vegetation), '
    'and peak heat index for each of the top-N exposures.',
    S_BODY))

top_cols = ['Rank', 'ID', 'Name', 'Type', 'Peak °C',
            'Imperv %', 'Veg %', 'HI °C', 'Hrs > SLA']
top_rows = []
for _, r in top_n.iterrows():
    def _f(v, fmt='{:.1f}'):
        return fmt.format(v) if pd.notna(v) else '—'
    top_rows.append([
        f'#{int(r.temp_rank)}',
        r.property_id,
        _wrap(r['name'], S_BODY),
        r['type'],
        _f(r.peak_temp_c),
        _f(r.get('impervious_pct'), '{:.0f}'),
        _f(r.get('vegetation_pct'), '{:.0f}'),
        _f(r.get('peak_heat_index_c')),
        _f(r.get('hours_above_sla'), '{:.0f}'),
    ])
top_table = Table([top_cols] + top_rows, repeatRows=1, hAlign='LEFT',
                  colWidths=[0.45*inch, 0.55*inch, 1.6*inch, 0.85*inch,
                             0.65*inch, 0.65*inch, 0.55*inch, 0.55*inch, 0.7*inch])
top_table.setStyle(TABLE_STYLE)
flowables.append(Spacer(1, 0.15 * inch))
flowables.append(top_table)
flowables.append(Spacer(1, 0.4 * inch))
flowables.append(CondPageBreak(4 * inch))

# Surface composition (C2 / C2b)
flowables.append(_h1('Surface composition across top-N (C2 / C2b)'))
flowables.append(Paragraph(
    'Stacked bars show the per-property class breakdown for satellite (overhead) '
    'and street view (front-of-building). Together they pin down whether the heat '
    'driver is impervious surface, low canopy, or a sky-exposed approach.',
    S_BODY))

c2_fig = _build_top_n_surface_comp_fig(seg_data if 'seg_data' in dir() else None,
                                        'C2 — Satellite surface composition (top-N)')
if c2_fig is not None:
    flowables.append(Spacer(1, 0.1 * inch))
    flowables.append(_fig_to_image(c2_fig, max_width_inches=USABLE_W / inch))

c2b_fig = None
if 'sv_data' in dir() and sv_data:
    sv_segs = {pid: d.get('segs', {}) for pid, d in sv_data.items()}
    c2b_fig = _build_top_n_surface_comp_fig(sv_segs,
                                             'C2b — Street-view scene composition (top-N)')
if c2b_fig is not None:
    flowables.append(Spacer(1, 0.2 * inch))
    flowables.append(_fig_to_image(c2b_fig, max_width_inches=USABLE_W / inch))
flowables.append(Spacer(1, 0.4 * inch))

# Per-property deep dive — each property starts a fresh page; its sub-sections
# (satellite / street-view / env-params + recommendation) flow continuously.
for _, r in top_n.iterrows():
    pid = r.property_id
    flowables.append(PageBreak())
    flowables.append(_h1(f"#{int(r.temp_rank)} · {pid} — {r['name']}"))

    metric_lines = [
        f"<b>Type:</b> {r['type']} &nbsp;·&nbsp; "
        f"<b>Peak temp:</b> {r.peak_temp_c:.1f} °C"
        + (f" @ {int(r.peak_hour):02d}:00" if pd.notna(r.peak_hour) else " (daily peak)")
        + (f" &nbsp;·&nbsp; <b>AOI percentile:</b> {r.aoi_percentile:.0f}"
           if pd.notna(r.get('aoi_percentile')) else ""),
    ]
    if pd.notna(r.get('impervious_pct')) or pd.notna(r.get('vegetation_pct')):
        metric_lines.append(
            f"<b>Surface:</b> impervious "
            f"{r.impervious_pct if pd.notna(r.impervious_pct) else '—'}% &nbsp;·&nbsp; "
            f"vegetation {r.vegetation_pct if pd.notna(r.vegetation_pct) else '—'}%"
        )
    if pd.notna(r.get('peak_heat_index_c')):
        metric_lines.append(
            f"<b>Peak heat index:</b> {r.peak_heat_index_c:.1f} °C "
            f"&nbsp;·&nbsp; <b>Hours above SLA:</b> "
            f"{int(r.hours_above_sla) if pd.notna(r.hours_above_sla) else '—'}"
        )
    for line in metric_lines:
        flowables.append(Paragraph(line, S_BODY))
    flowables.append(Spacer(1, 0.1 * inch))

    # Satellite — side-by-side pair.
    if pid in (sat_imgs or {}):
        imgs = sat_imgs[pid]
        orig_pil = _decode_b64(imgs.get('orig'))
        seg_pil  = _decode_b64(imgs.get('seg'))
        if orig_pil is not None or seg_pil is not None:
            flowables.append(_h2('Satellite — original & segmentation'))
            row_imgs = []
            if orig_pil is not None:
                row_imgs.append([_pil_to_image(orig_pil, max_width_inches=3.1),
                                 Paragraph('original satellite tile', S_CAP)])
            else:
                row_imgs.append([Paragraph('(original missing)', S_CAP)])
            if seg_pil is not None:
                row_imgs.append([_pil_to_image(seg_pil, max_width_inches=3.1),
                                 Paragraph('segmentation overlay', S_CAP)])
            else:
                row_imgs.append([Paragraph('(segmentation missing)', S_CAP)])
            sat_row = Table([[c[0] for c in row_imgs], [c[1] for c in row_imgs]],
                            colWidths=[3.3 * inch, 3.3 * inch])
            sat_row.setStyle(TableStyle([
                ('VALIGN', (0, 0), (-1, -1), 'TOP'),
                ('ALIGN',  (0, 0), (-1, -1), 'CENTER'),
                ('LEFTPADDING', (0, 0), (-1, -1), 0),
                ('RIGHTPADDING',(0, 0), (-1, -1), 0),
                ('TOPPADDING',  (0, 0), (-1, -1), 0),
                ('BOTTOMPADDING',(0, 0), (-1, -1), 0),
            ]))
            flowables.append(sat_row)
            flowables.append(Spacer(1, 0.1 * inch))

    if pid in (seg_data or {}) and seg_data[pid]:
        cb_fig = _build_seg_breakdown_fig(seg_data[pid],
                                           f'Satellite class breakdown — {pid}',
                                           color='#3b8686')
        if cb_fig is not None:
            flowables.append(_fig_to_image(cb_fig, max_width_inches=USABLE_W / inch))
    flowables.append(Spacer(1, 0.3 * inch))
    flowables.append(CondPageBreak(4 * inch))

    # Street view — side-by-side pair matching the satellite section.
    flowables.append(_h1(f"#{int(r.temp_rank)} · {pid} — {r['name']} (street view)"))
    if pid in (sv_data or {}):
        d = sv_data[pid]
        orig_pil = _decode_b64(d.get('orig'))
        seg_pil  = _decode_b64(d.get('seg'))
        flowables.append(Paragraph(f"Imagery date: {d.get('image_date', 'n/a')}", S_BODY))
        if orig_pil is not None or seg_pil is not None:
            cells = []
            if orig_pil is not None:
                cells.append([_pil_to_image(orig_pil, max_width_inches=3.1),
                              Paragraph('original street-view (front)', S_CAP)])
            else:
                cells.append([Paragraph('(original missing)', S_CAP)])
            if seg_pil is not None:
                cells.append([_pil_to_image(seg_pil, max_width_inches=3.1),
                              Paragraph('pixel-wise segmentation', S_CAP)])
            else:
                cells.append([Paragraph('(segmentation missing)', S_CAP)])
            sv_row = Table([[c[0] for c in cells], [c[1] for c in cells]],
                           colWidths=[3.3 * inch, 3.3 * inch])
            sv_row.setStyle(TableStyle([
                ('VALIGN', (0, 0), (-1, -1), 'TOP'),
                ('ALIGN',  (0, 0), (-1, -1), 'CENTER'),
                ('LEFTPADDING',  (0, 0), (-1, -1), 0),
                ('RIGHTPADDING', (0, 0), (-1, -1), 0),
                ('TOPPADDING',   (0, 0), (-1, -1), 0),
                ('BOTTOMPADDING',(0, 0), (-1, -1), 0),
            ]))
            flowables.append(sv_row)
            flowables.append(Spacer(1, 0.1 * inch))
        if d.get('segs'):
            sv_cb_fig = _build_seg_breakdown_fig(d['segs'],
                                                  f'Street-view class breakdown — {pid}',
                                                  color='#7a5195')
            if sv_cb_fig is not None:
                flowables.append(_fig_to_image(sv_cb_fig, max_width_inches=USABLE_W / inch))
    flowables.append(Spacer(1, 0.3 * inch))
    flowables.append(CondPageBreak(3.5 * inch))

    # Env-params diurnal
    flowables.append(_h1(f"#{int(r.temp_rank)} · {pid} — {r['name']} (env-params)"))
    s_env = (env_data or {}).get(pid)
    if s_env and s_env.get('heat_index_celsius'):
        flowables.append(Paragraph(
            f"Diurnal drivers across the day — heat index, apparent temperature, RH, "
            f"and solar irradiance. SLA threshold ({SLA_HI_C} °C) marked.",
            S_BODY))
        ev_fig = _build_diurnal_fig(s_env,
                                     f"Diurnal drivers — #{int(r.temp_rank)} {pid}")
        if ev_fig is not None:
            flowables.append(_fig_to_image(ev_fig, max_width_inches=USABLE_W / inch))
    else:
        flowables.append(Paragraph('No env-params data available for this property.', S_BODY))

    # Recommended action
    action_label, evidence, program = _pick_action(r)
    flowables.append(Spacer(1, 0.15 * inch))
    flowables.append(_h2('Recommended action'))
    flowables.append(Paragraph(f"<b>→ {action_label}</b>", S_BODY))
    flowables.append(Paragraph(evidence, S_BODY))
    flowables.append(Paragraph(f"<i>Program: {program}</i>", S_BODY))

# Consolidated action briefs
flowables.append(PageBreak())
flowables.append(_h1('Action briefs — top-N consolidated'))
for _, r in top_n.iterrows():
    rank = int(r.temp_rank)
    action_label, evidence, program = _pick_action(r)
    peak_str = f"{r.peak_temp_c:.1f} °C"
    if pd.notna(r.peak_hour):
        peak_str += f" @ {int(r.peak_hour):02d}:00"
    flowables.append(Paragraph(
        f"#{rank} · {r['property_id']} — {r['name']} "
        f"<font color='#5a6b7b'>({r['type']} · peak {peak_str})</font>",
        S_H2))
    flowables.append(Paragraph(f"<b>→ {action_label}</b> — {evidence}", S_BODY))
    flowables.append(Paragraph(f"<i>Program: {program}</i>", S_BODY))
    flowables.append(Spacer(1, 0.1 * inch))
flowables.append(Spacer(1, 0.3 * inch))
flowables.append(CondPageBreak(4 * inch))

# Full portfolio audit table
flowables.append(_h1(f'Full portfolio audit ({len(portfolio)} properties)'))
flowables.append(Paragraph(
    'Every property in the portfolio with its measured peak temperature and AOI percentile. '
    'Properties not in the top-N receive heatmap-derived signals only — surface diagnosis and '
    'env-params columns are populated only for the top-N exposures (Steps 6–8).',
    S_BODY))

audit_cols = ['Rank', 'ID', 'Name', 'Type', 'Peak °C',
              'Imperv %', 'Veg %', 'HI °C', 'AOI %ile']
audit_rows = []
for _, p in portfolio.iterrows():
    def _f(v, fmt='{:.1f}'):
        return fmt.format(v) if pd.notna(v) else '—'
    audit_rows.append([
        int(p.temp_rank),
        p.property_id,
        _wrap(p['name'], S_BODY_W),
        p['type'],
        _f(p.peak_temp_c),
        _f(p.get('impervious_pct'), '{:.0f}'),
        _f(p.get('vegetation_pct'), '{:.0f}'),
        _f(p.get('peak_heat_index_c')),
        _f(p.get('aoi_percentile'), '{:.0f}'),
    ])
audit_table = Table([audit_cols] + audit_rows, repeatRows=1, hAlign='LEFT',
                    colWidths=[0.45*inch, 0.55*inch, 1.85*inch, 0.85*inch,
                               0.6*inch, 0.6*inch, 0.55*inch, 0.55*inch, 0.65*inch])
audit_table.setStyle(TABLE_STYLE)
flowables.append(Spacer(1, 0.15 * inch))
flowables.append(audit_table)

# Closing contact block — yellow rule, contact heading + text, centered blue wordmark.
yellow_rule = Table([['']], colWidths=[USABLE_W], rowHeights=[0.06 * inch])
yellow_rule.setStyle(TableStyle([
    ('BACKGROUND',    (0, 0), (-1, -1), BRAND_YELLOW),
    ('LEFTPADDING',   (0, 0), (-1, -1), 0),
    ('RIGHTPADDING',  (0, 0), (-1, -1), 0),
    ('TOPPADDING',    (0, 0), (-1, -1), 0),
    ('BOTTOMPADDING', (0, 0), (-1, -1), 0),
]))

closing_block = [
    Spacer(1, 0.45 * inch),
    yellow_rule,
    Spacer(1, 0.25 * inch),
    _h1('FortyGuard contact details'),
    Paragraph(
        'FortyGuard Tech Limited, Al Khatem Tower, 14th Floor, 112, WeWork Hub71, '
        'Abu Dhabi Global Market Square, Al Maryah Island, Abu Dhabi, UAE. '
        'PoBox: 3317, Tel: +97126662799, Email: info@fortyguard.com',
        S_CONTACT),
    Spacer(1, 0.35 * inch),
]

if LOGO_FOOTER_PATH.exists():
    closing_logo_w = 1.8 * inch
    closing_logo_h = closing_logo_w * LOGO_FOOTER_ASPECT
    closing_logo   = RLImage(str(LOGO_FOOTER_PATH),
                             width=closing_logo_w, height=closing_logo_h)
    closing_logo.hAlign = 'CENTER'
    closing_block.append(closing_logo)

flowables.append(KeepTogether(closing_block))


# ── Render the PDF. ───────────────────────────────────────────────────────
pdf_path = OUT_DIR / 'portfolio_report.pdf'
doc = SimpleDocTemplate(
    str(pdf_path),
    pagesize=letter,
    leftMargin=LEFT_MARGIN, rightMargin=RIGHT_MARGIN,
    topMargin=TOP_MARGIN,   bottomMargin=BOT_MARGIN,
    title=f'{REPORT_NAME} · {STUDY_DATE}',
    author='FortyGuard',
)
doc.build(flowables, onFirstPage=_draw_cover, onLaterPages=_draw_body)

print(f'  ✓ {pdf_path.relative_to(ROOT)}')
print(f'\nFull bundle: {OUT_DIR.relative_to(ROOT)}')
print(f'  - portfolio_evaluation.csv')
print(f'  - portfolio_report.pdf')
print(f'  - maps/*.html  (open any in a browser)')


---
## Wrap-up

Starting from a real-estate portfolio CSV you now have evidence-based answers for the client meeting:

| Artifact | Used by |
|----------|---------|
| **M1** portfolio overview map | Client deck — slide 1 |
| **M2** hot-exposures zoom map | Client deck — slide 2 |
| **M3** final ranked priority map | Client deck — slide 3 |
| **C1** diurnal temperature curves | Backup / analyst review |
| **C2** surface composition stacked bar (top-N) | Backup |
| **C3** HI / apparent / RH / solar profile per property | Deep-dive slide |
| **C4** risk vs opportunity scatter (4 quadrants) | Decision slide |
| Per-property action brief HTML cards | Asset-management hand-off |
| **Bundled hand-off folder (CSV + PDF report + interactive maps)** | **Slide decks, client packet, ops** |

### Where everything lives on disk

```
data/
  heatmaps/       ← raw heatmap GeoJSON outputs (live + cached)
  satellite/      ← raw satellite-segmentation JSON per property (live + cached)
  street_view/    ← raw street-view JSON per property (live + cached)
  env_params/     ← raw env-params JSON per property (live + cached)
outputs/
  real_estate_<STUDY_DATE>/
    portfolio_evaluation.csv
    portfolio_report.pdf      ← multi-page PDF for the client deck
    maps/*.html               ← interactive folium maps (M1, M2, M3 + per-step diagnostic maps)
```

Re-running against any captured live response is a one-liner: change the filename in the matching cache cell (Step 2b / 6b / 7b / 8b) to point at any file in the corresponding `data/<type>/` subfolder.

Every input is explicit. Every weight is at the top of the notebook. Run Steps 2a / 6a / 7a / 8a (or just the ones you need) with a `FORTYGUARD_API_KEY` to refresh against today's data; run the matching `b` cells to replay any captured run offline.

**Apply this pattern to adjacent use cases**: insured-properties portfolio (insurance underwriting), data-center sites (operational-risk screening), hospitality assets (guest-comfort benchmarking), retail acquisitions (foot-traffic comfort). The workflow — *portfolio × diurnal heatmap × surface diagnosis × ground-truth × env-params → risk + opportunity table* — transfers directly.